In [8]:
"""
===============================================================================
COMPLETE ARABIC LAW EXTRACTION SYSTEM - ALL-IN-ONE NOTEBOOK
===============================================================================

This notebook contains the complete improved extraction system in a single file.
Designed for ~100% extraction accuracy on Arabic legal texts.

Author: Claude (Anthropic)
Date: February 2026
Version: 2.0 (Improved)

USAGE:
1. Set your paths in the CONFIGURATION section
2. Run all cells in order
3. Review validation reports
4. Check quality scores (target: >90)

===============================================================================
"""

# %%
# =============================================================================
# SECTION 1: IMPORTS AND SETUP
# =============================================================================

from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple, Set, Iterable, Union
from pathlib import Path
from collections import Counter
import hashlib
import re
import logging
import traceback
import json
import pandas as pd

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
%cd /media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject
print("✓ Imports complete")

# %%
# =============================================================================
# SECTION 2: HELPER FUNCTIONS
# =============================================================================

def _stable_id(s: str) -> str:
    """Generate a stable hash ID for a string."""
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]


def _dedup_rules(rules: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Remove duplicate rules while preserving order."""
    seen = set()
    out = []
    for r in rules:
        key = (r.get("type"), str(r.get("article_no")), r.get("text"))
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out

print("✓ Helper functions defined")

# %%
# =============================================================================
# SECTION 3: DATA STRUCTURES
# =============================================================================

@dataclass
class ExtractedLaw:
    """Container for all extracted law components."""
    law_meta: Dict[str, Any]
    articles: List[Dict[str, Any]]          # [{article_no, text}]
    terms: List[Dict[str, Any]]             # [{term, definition, article_no}]
    topics: List[Dict[str, Any]]            # [{topic, article_no}]
    references: List[Dict[str, Any]]        # [{from_article_no, to_article_no}]
    rules: List[Dict[str, Any]]             # [{type, article_no, text}]

print("✓ Data structures defined")

# %%
# =============================================================================
# SECTION 4: BASE EXTRACTOR
# =============================================================================

class BaseLawExtractor:
    """
    Enhanced base extractor with improved Arabic text handling.
    أدنى مستوى = المادة (Article)
    """

    # Enhanced article marker pattern
    ARTICLE_MARK_RE = re.compile(
        r"""(?m)^\s*
            (?:المادة|مادة)\s*              # article keyword (required)
            (?:\(\s*)?                      # optional opening paren
            (?P<num>
                [0-9٠-٩]+                   # base number
                (?:
                    \s*
                    (?:
                        مكرر(?:[اأإآى])?    # مكرر with optional suffix
                        (?:\s*(?:[اأإآA-Za-z]))?
                        |
                        (?:أولا?ً?|ثانيا?ً?|ثالثا?ً?|رابعا?ً?|خامسا?ً?)
                    )
                )?
                (?:\s*\(\s*[اأإآA-Za-z]\s*\))?
            )
            (?:\s*\))?                      # optional closing paren
            \s*[:：\-–\s]                   # separator
            (?P<rest>.*)
            $
        """,
        re.VERBOSE
    )

    # Enhanced reference pattern
    REF_RE = re.compile(
        r"""
        (?:المادة|مادة|المواد)\s*
        (?:\(\s*)?
        (?P<num>
            [0-9٠-٩]+
            (?:\s*(?:مكرر(?:[اأإآى])?)?)?
            (?:\s*\(\s*[اأإآA-Za-z]\s*\))?
        )
        (?:\s*\))?
        (?:
            \s*(?:و|،|,)\s*
            (?:\(\s*)?
            (?P<num2>[0-9٠-٩]+(?:\s*مكرر(?:[اأإآى])?)?)
            (?:\s*\))?
        )?
        """,
        re.VERBOSE
    )

    # Enhanced definition pattern
    DEF_LINE_RE = re.compile(
        r"""
        (?mx)
        ^\s*
        (?:
            (?:في|لأغراض?|بموجب|وفقا?\s+ل?|طبقا?\s+ل?)\s+
            (?:هذا\s+)?
            (?:القانون|النظام|اللائحة|المرسوم)?
            [,،:]?\s*
        )?
        (?:
            يُ?قصد\s+(?:ب|من)|
            تعني|يعني|المقصود\s+(?:ب|من)|
            يُ?عرّ?ف(?:\s+على\s+أنه?)?|
            يُ?راد\s+ب|
            يُ?شار\s+إليه?|
            يُ?طلق\s+على?|
            هو|هي|
            means?|
            is\s+defined\s+as|
            shall\s+mean|
            refers?\s+to|
            definition\s+of
        )
        \s+
        (?P<term>
            [\(\[\{]?\s*
            ["'«""`]?
            [^\n:：\-–]{1,200}?
            ["'»""`]?
            \s*[\)\]\}]?
        )
        \s*[:：\-–]\s*
        (?P<def>.+)
        $
        """,
        re.VERBOSE | re.IGNORECASE
    )

    _ARABIC_INDIC_TRANS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

    def __init__(self, raw_text: str, law_id: str, law_name: str):
        self.raw_text = raw_text or ""
        self._law_id = law_id
        self._law_name = law_name
        self._validation_warnings: List[str] = []

    def _to_western_digits(self, s: str) -> str:
        """Convert Arabic-Indic numerals to Western."""
        return s.translate(self._ARABIC_INDIC_TRANS)

    def _normalize_article_no(self, token: str) -> str:
        """Normalize article number to standard format."""
        token = token.strip()
        token = self._to_western_digits(token)
        token = re.sub(r"\s+", " ", token)
        token = re.sub(r"مكر(?:ر[اأإآىًٍَُِّْ]?)", "مكرر", token)
        
        ordinal_map = {
            "أولاً": "أولا", "أولا": "أولا", "أول": "أولا",
            "ثانياً": "ثانيا", "ثانيا": "ثانيا", "ثاني": "ثانيا",
            "ثالثاً": "ثالثا", "ثالثا": "ثالثا", "ثالث": "ثالثا",
            "رابعاً": "رابعا", "رابعا": "رابعا", "رابع": "رابعا",
        }
        for old, new in ordinal_map.items():
            token = token.replace(old, new)
        
        return token.strip()

    def extract_law_meta(self) -> Dict[str, Any]:
        """Extract law metadata."""
        return {
            "law_id": self._law_id,
            "name": self._law_name,
            "text_length": len(self.raw_text),
            "validation_warnings": self._validation_warnings
        }

    def extract_articles(self) -> List[Dict[str, str]]:
        """Extract articles with improved accuracy."""
        text = self.raw_text
        matches = list(self.ARTICLE_MARK_RE.finditer(text))

        articles: List[Dict[str, str]] = []

        if not matches:
            logger.warning("No article markers found in text")
            self._validation_warnings.append("No article markers found")
            return [{
                "article_no": None,
                "text": text.strip()
            }]

        logger.info(f"Found {len(matches)} article markers")

        # Extract preamble
        first = matches[0]
        if first.start() > 0:
            preamble = text[:first.start()].strip()
            if preamble:
                articles.append({
                    "article_no": None,
                    "text": preamble
                })
                logger.debug(f"Extracted preamble ({len(preamble)} chars)")

        # Extract each article
        for i, m in enumerate(matches):
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

            body = text[start:end].strip()
            raw_no = m.group("num") or ""
            article_no = self._normalize_article_no(raw_no)

            if len(body) < 10 and i < len(matches) - 1:
                logger.warning(f"Article {article_no} is very short ({len(body)} chars)")
                self._validation_warnings.append(f"Short article: {article_no}")

            articles.append({
                "article_no": article_no,
                "text": body,
                "raw_article_no": raw_no
            })
            
            logger.debug(f"Extracted article {article_no} ({len(body)} chars)")

        logger.info(f"Total extracted: {len(articles)} articles")
        return articles

    def extract_references(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Extract internal article references."""
        refs = set()
        
        for a in articles:
            from_no = str(a.get("article_no", "")).strip()
            if not from_no or from_no == "None":
                continue
                
            text = a.get("text", "")
            
            for match in self.REF_RE.finditer(text):
                to_no = match.group("num")
                if to_no:
                    to_no = self._normalize_article_no(to_no)
                    if to_no and to_no != from_no:
                        refs.add((from_no, to_no))
                
                to_no2 = match.group("num2") if match.lastindex >= 2 else None
                if to_no2:
                    to_no2 = self._normalize_article_no(to_no2)
                    if to_no2 and to_no2 != from_no:
                        refs.add((from_no, to_no2))
        
        result = [{"from_article_no": f, "to_article_no": t} for f, t in sorted(refs)]
        logger.info(f"Extracted {len(result)} references")
        return result

    def extract_terms_definitions(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Extract term definitions with multi-line handling."""
        out = []
        
        for a in articles:
            article_no = a.get("article_no")
            text = a.get("text", "")
            
            # Line-by-line
            for line in text.splitlines():
                line = line.strip()
                if not line or len(line) < 10:
                    continue
                    
                m = self.DEF_LINE_RE.search(line)
                if m:
                    term = m.group("term").strip(" \t\n\r-–:：\"""'«»()[]")
                    definition = m.group("def").strip()
                    term = re.sub(r'\s+', ' ', term)
                    
                    if term and definition and len(term) > 1:
                        out.append({
                            "term": term,
                            "definition": definition,
                            "article_no": article_no
                        })
                        logger.debug(f"Found definition: {term[:50]}... in article {article_no}")
            
            # Multi-line definitions
            multi_line_pattern = re.compile(
                r"(?:يُ?قصد\s+ب|تعني|المقصود\s+ب)\s*([^:\n]{5,100})\s*[:：]\s*([^\n]+(?:\n(?!\s*(?:المادة|مادة|\d+[.)]))[^\n]+)*)",
                re.MULTILINE
            )
            
            for m in multi_line_pattern.finditer(text):
                term = m.group(1).strip(" \t\n\r-–:：\"""'«»()[]")
                definition = m.group(2).strip()
                definition = re.sub(r'\s+', ' ', definition)
                
                if term and definition and len(term) > 1:
                    if not any(d["term"] == term and d["article_no"] == article_no for d in out):
                        out.append({
                            "term": term,
                            "definition": definition,
                            "article_no": article_no
                        })
                        logger.debug(f"Found multi-line definition: {term[:50]}...")
        
        logger.info(f"Extracted {len(out)} term definitions")
        return out

    def extract_topics(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Override in subclass."""
        return []

    def extract_rules(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Override in subclass."""
        return []

    def extract_all(self) -> ExtractedLaw:
        """Extract all components."""
        logger.info(f"Starting extraction for {self._law_name}")
        
        meta = self.extract_law_meta()
        articles = self.extract_articles()
        references = self.extract_references(articles)
        terms = self.extract_terms_definitions(articles)
        topics = self.extract_topics(articles)
        rules = self.extract_rules(articles)

        logger.info(f"Extraction complete: {len(articles)} articles, {len(terms)} terms, "
                   f"{len(references)} refs, {len(topics)} topics, {len(rules)} rules")

        return ExtractedLaw(
            law_meta=meta,
            articles=articles,
            terms=terms,
            topics=topics,
            references=references,
            rules=rules,
        )

print("✓ Base extractor defined")

# %%
# =============================================================================
# SECTION 5: MIXINS (REUSABLE COMPONENTS)
# =============================================================================

class PenalRulesMixin:
    """Enhanced penal rule extractor."""

    _AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
    _TATWEEL_RE = re.compile(r"\u0640")

    @staticmethod
    def _normalize(text: str) -> str:
        """Normalize Arabic text for pattern matching."""
        if not text:
            return ""
        text = PenalRulesMixin._AR_DIACRITICS_RE.sub("", text)
        text = PenalRulesMixin._TATWEEL_RE.sub("", text)
        replacements = [
            ("أ", "ا"), ("إ", "ا"), ("آ", "ا"),
            ("ى", "ي"), ("ئ", "ي"),
            ("ة", "ه"), ("ؤ", "و")
        ]
        for old, new in replacements:
            text = text.replace(old, new)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    PENALTY_LINE_RE = re.compile(
        r"""(?ix).*?\b(يُ?عاقب(?:\s+على)?(?:\s+ب)?|تُ?عاقب|يُ?حكم\s+(?:عليه?|ب)|الحكم\s+(?:عليه?|ب)|تكون\s+العقوبه?|يُ?عاقب\s+ب|العقوبه?\s+(?:هي|السجن|الحبس|الغرامه?|الاعدام|الاشغال)|يُ?حبس|يُ?سجن|يُ?غرّ?م|عقوبت?ه?)\b.*""",
        re.VERBOSE
    )

    CRIME_HINT_RE = re.compile(
        r"""(?ix).*?\b(كل\s+من|من\s+(?:ارتكب|يرتكب|قام\s+ب|يقوم\s+ب)|يُ?عد(?:ّ?)?\s+.*?جريم?ه?|يُ?عتبر\s+.*?جريم?ه?|يُ?رتكب\s+جريم?ه?|جريم?ه?\s+.*?(?:ارتكاب|القيام)|جنايه?|جنحه?|مخالفه?)\b.*""",
        re.VERBOSE
    )

    MEASURE_HINT_RE = re.compile(
        r"""(?ix)\b(مصادر?ه?|مُ?صادر|(?:ا|إ)?غلاق|غلق|تجميد|حجب|(?:ا|إ)?دراج|التحفظ\s+على?|منع|حظر|سحب\s+(?:الترخيص|الرخصه?)|(?:ا|إ)?لغاء\s+(?:الترخيص|الرخصه?)|وقف\s+النشاط|مراقبه?|ضبط)\b""",
        re.VERBOSE
    )

    def extract_penal_like_rules(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Extract penal rules."""
        rules: List[Dict[str, Any]] = []
        seen: Set[Tuple[str, str, str]] = set()

        for article in articles:
            art_no = str(article.get("article_no", "")).strip()
            text = article.get("text", "")
            if not text:
                continue

            chunks = self._split_into_chunks(text)

            for chunk in chunks:
                if not chunk or len(chunk) < 15:
                    continue

                raw_chunk = chunk
                normalized = self._normalize(chunk)

                if self.PENALTY_LINE_RE.match(normalized):
                    key = ("PenaltyRule", art_no, normalized[:200])
                    if key not in seen:
                        seen.add(key)
                        rules.append({"type": "PenaltyRule", "article_no": art_no, "text": raw_chunk})

                if self.CRIME_HINT_RE.match(normalized):
                    key = ("CrimeRule", art_no, normalized[:200])
                    if key not in seen:
                        seen.add(key)
                        rules.append({"type": "CrimeRule", "article_no": art_no, "text": raw_chunk})

                if self.MEASURE_HINT_RE.search(normalized):
                    key = ("MeasureRule", art_no, normalized[:200])
                    if key not in seen:
                        seen.add(key)
                        rules.append({"type": "MeasureRule", "article_no": art_no, "text": raw_chunk})

        logger.info(f"Extracted {len(rules)} penal rules")
        return rules

    @staticmethod
    def _split_into_chunks(text: str) -> List[str]:
        """Split text into logical chunks."""
        chunks = [p.strip() for p in text.split('\n\n') if p.strip()]
        result = []
        for chunk in chunks:
            if len(chunk) > 500:
                result.extend([line.strip() for line in chunk.split('\n') if line.strip()])
            else:
                result.append(chunk)
        return result if result else [text]


class ProcedureRulesMixin:
    """Enhanced procedure rule extractor."""

    _AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
    _TATWEEL_RE = re.compile(r"\u0640")

    @staticmethod
    def _normalize_ar(text: str) -> str:
        """Normalize Arabic text."""
        if not text:
            return ""
        text = ProcedureRulesMixin._AR_DIACRITICS_RE.sub("", text)
        text = ProcedureRulesMixin._TATWEEL_RE.sub("", text)
        replacements = [
            ("أ", "ا"), ("إ", "ا"), ("آ", "ا"),
            ("ى", "ي"), ("ئ", "ي"),
            ("ؤ", "و"), ("ة", "ه")
        ]
        for old, new in replacements:
            text = text.replace(old, new)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    PROC_LINE_RE = re.compile(
        r"""(?ix).*?\b(?:يجب(?:\s+على?|\s+أن)?|يتعيّ?ن(?:\s+على?|\s+أن)?|يلزم(?:\s+على?|\s+أن)?|يلتزم(?:\s+ب)?|على\s+[\w\s]{1,60}?\s+أن|يتحتّ?م|يترتّ?ب\s+على?|يجوز(?:\s+ل)?|يُ?سمح(?:\s+ب)?|يُ?مكن|يحقّ?(?:\s+ل)?|لا\s+يجوز|لا\s+يُ?سمح|لا\s+يُ?مكن|يُ?حظر|يُ?منع|محظور|ممنوع|يشترط|بشرط\s+أن|شريطه?\s+أن|خاضع(?:\s+ل)?|مرهون(?:\s+ب)?|موقوف(?:\s+على)?)\b.*""",
        re.VERBOSE
    )

    PROC_LINE_EN_RE = re.compile(
        r"""(?ix).*?\b(?:must|shall|should|may\s+(?:not|only)?|is\s+(?:required|mandatory|obligatory|prohibited|forbidden|permitted)|shall\s+(?:not|be)|required\s+(?:to|that)|subject\s+to|conditional\s+(?:on|upon)|prohibited|forbidden)\b.*""",
        re.VERBOSE
    )

    def extract_procedure_rules(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Extract procedure rules."""
        out: List[Dict[str, Any]] = []
        seen: Set[Tuple[str, str, str]] = set()

        for a in articles:
            art_no = a.get("article_no", "")
            text = a.get("text", "") or ""
            if not text:
                continue

            chunks = self._split_text_chunks(text)

            for chunk in chunks:
                if not chunk or len(chunk) < 15:
                    continue

                normalized = self._normalize_ar(chunk)
                matched = False

                if self.PROC_LINE_RE.search(normalized):
                    matched = True
                elif self.PROC_LINE_EN_RE.search(chunk):
                    matched = True
                elif re.search(r"على\s+[\w\s]{1,60}?\s+أن", chunk):
                    matched = True

                if matched:
                    key = ("ProcedureRule", str(art_no), normalized[:200])
                    if key not in seen:
                        seen.add(key)
                        out.append({"type": "ProcedureRule", "article_no": art_no, "text": chunk})

        logger.info(f"Extracted {len(out)} procedure rules")
        return out

    @staticmethod
    def _split_text_chunks(text: str) -> List[str]:
        """Split text into chunks."""
        chunks = []
        buffer = []
        
        for line in text.splitlines():
            line = line.strip()
            if not line:
                if buffer:
                    chunks.append(" ".join(buffer))
                    buffer = []
                continue
            
            if len(line) > 200:
                sentences = re.split(r'[.。]\s+', line)
                for sent in sentences:
                    sent = sent.strip()
                    if sent:
                        if len(sent) > 300:
                            buffer.append(sent)
                            chunks.append(" ".join(buffer))
                            buffer = []
                        else:
                            buffer.append(sent)
            else:
                buffer.append(line)
        
        if buffer:
            chunks.append(" ".join(buffer))
        
        return chunks if chunks else [text]


class TopicsMixin:
    """Enhanced topic extractor."""

    TOPIC_MAP: Dict[Union[str, Tuple[str, ...]], str] = {}

    _AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
    _TATWEEL_RE = re.compile(r"\u0640")

    @staticmethod
    def _normalize_ar(text: str) -> str:
        """Normalize Arabic text."""
        if not text:
            return ""
        text = TopicsMixin._AR_DIACRITICS_RE.sub("", text)
        text = TopicsMixin._TATWEEL_RE.sub("", text)
        replacements = [
            ("أ", "ا"), ("إ", "ا"), ("آ", "ا"),
            ("ى", "ي"), ("ئ", "ي"),
            ("ؤ", "و"), ("ة", "ه")
        ]
        for old, new in replacements:
            text = text.replace(old, new)
        text = re.sub(r"\s+", " ", text).strip().lower()
        return text

    @staticmethod
    def _iter_keys(mapping_key: Union[str, Tuple[str, ...]]) -> Iterable[str]:
        """Iterate over keys."""
        if isinstance(mapping_key, (list, tuple)):
            for k in mapping_key:
                yield k
        else:
            yield mapping_key

    def extract_topics_by_map(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Extract topics using TOPIC_MAP."""
        pairs: Set[Tuple[str, str]] = set()

        prepared: List[Tuple[re.Pattern, str]] = []
        
        for raw_key, topic in self.TOPIC_MAP.items():
            for key in self._iter_keys(raw_key):
                if not key:
                    continue
                
                if isinstance(key, str) and key.startswith(("re:", "RE:", "regex:")):
                    pat_str = key.split(":", 1)[1]
                    try:
                        compiled = re.compile(pat_str, re.IGNORECASE | re.UNICODE)
                    except re.error as e:
                        logger.warning(f"Invalid regex pattern '{pat_str}': {e}")
                        compiled = re.compile(re.escape(pat_str), re.IGNORECASE | re.UNICODE)
                else:
                    nk = self._normalize_ar(key)
                    compiled = re.compile(r"\b" + re.escape(nk) + r"\b", re.IGNORECASE | re.UNICODE)
                
                prepared.append((compiled, topic))

        for a in articles:
            art_no = a.get("article_no", "")
            if not art_no or str(art_no) == "None":
                continue
                
            txt = a.get("text", "") or ""
            if not txt:
                continue
            
            norm_txt = self._normalize_ar(txt)

            for pat, topic in prepared:
                try:
                    if pat.search(norm_txt):
                        pairs.add((topic, str(art_no)))
                except re.error:
                    try:
                        if pat.search(txt):
                            pairs.add((topic, str(art_no)))
                    except re.error as e:
                        logger.warning(f"Pattern matching error for topic '{topic}': {e}")

        result = [
            {"topic": t, "article_no": no} 
            for t, no in sorted(pairs, key=lambda x: (str(x[0]), str(x[1])))
        ]
        
        logger.info(f"Extracted {len(result)} topic associations")
        return result

print("✓ Mixins defined")

# %%
# =============================================================================
# SECTION 6: LAW-SPECIFIC EXTRACTORS
# =============================================================================

class PenalCodeExtractor(BaseLawExtractor, PenalRulesMixin, TopicsMixin):
    """Penal Code extractor."""
    
    TOPIC_MAP = {
        ("يعاقب", "يعاقب عليه", "يعاقب ب", "يعاقب بالحبس", "يعاقب بالسجن"): "Penalty",
        ("العقوبة", "العقوبه", "عقوبة"): "Penalty",
        ("غرامة", "غرامه", "بغرامة"): "Penalty",
        ("حبس", "الحبس"): "Penalty",
        ("سجن", "السجن", "بالسجن"): "Penalty",
        ("إعدام", "اعدام", "الاعدام"): "Penalty",
        ("مصادرة", "مصادره", "المصادرة", "بمصادرة"): "Measure",
        ("إغلاق", "اغلاق", "الاغلاق"): "Measure",
        ("حجب", "الحجب"): "Measure",
        ("اشتراك", "الاشتراك", "شريك", "الشريك"): "Participation",
        ("محرض", "التحريض", "تحريض"): "Participation",
        ("شروع", "الشروع", "محاولة", "المحاولة", "شروعا"): "Attempt",
        ("عود", "العود", "العائد", "عائد"): "Recidivism",
    }

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="penal_code", law_name="قانون العقوبات")

    def extract_topics(self, articles: List[Dict[str, Any]]):
        return self.extract_topics_by_map(articles)

    def extract_rules(self, articles: List[Dict[str, Any]]):
        return _dedup_rules(self.extract_penal_like_rules(articles))


class MoneyLaunderingExtractor(BaseLawExtractor, PenalRulesMixin, TopicsMixin):
    """Money Laundering Law extractor."""
    
    TOPIC_MAP = {
        ("غسل", "غسل الاموال", "غسل الأموال", "غسيل", "غسيل الاموال"): "MoneyLaundering",
        ("المتحصلات", "العائدات", "متحصلات", "عائدات"): "Proceeds",
        ("تجميد", "التجميد"): "Freezing",
        ("مصادرة", "المصادرة", "مصادره"): "Confiscation",
        ("الإبلاغ", "ابلاغ", "البلاغ", "إبلاغ"): "Reporting",
        ("حظر الإفشاء", "حظر الافصاح", "حظر الافشاء", "الإفشاء"): "TippingOff",
        ("العناية الواجبة", "العناية الوجيبه", "العناية", "الواجبة"): "DueDiligence",
        ("التحقق", "التحقق من الهوية", "معرفة العميل"): "DueDiligence",
        ("يعاقب", "العقوبة", "عقوبة"): "Penalty",
    }

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="money_laundering", law_name="قانون مكافحة غسل الأموال")

    def extract_topics(self, articles: List[Dict[str, Any]]):
        return self.extract_topics_by_map(articles)

    def extract_rules(self, articles: List[Dict[str, Any]]):
        return _dedup_rules(self.extract_penal_like_rules(articles))


class WeaponsAmmunitionExtractor(BaseLawExtractor, PenalRulesMixin, TopicsMixin):
    """Weapons and Ammunition Law extractor."""
    
    TOPIC_MAP = {
        ("سلاح", "أسلحة", "اسلحة", "السلاح", "الأسلحة"): "Weapons",
        ("سلاح ناري", "اسلحة نارية", "أسلحة نارية"): "Firearms",
        ("ذخيرة", "ذخائر", "الذخيرة"): "Ammunition",
        ("ترخيص", "رخصة", "ترخيصات", "الترخيص", "الرخصة"): "Licensing",
        ("إحراز", "احراز", "الإحراز"): "Possession",
        ("حيازة", "الحيازة"): "Possession",
        ("اتجار", "اتجار بالاسلحة", "الاتجار"): "Trafficking",
        ("تهريب", "التهريب"): "Smuggling",
        ("مصادرة", "المصادرة"): "Confiscation",
        ("يعاقب", "العقوبة"): "Penalty",
    }

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="weapons_ammunition", law_name="قانون الأسلحة والذخيرة")

    def extract_topics(self, articles: List[Dict[str, Any]]):
        return self.extract_topics_by_map(articles)

    def extract_rules(self, articles: List[Dict[str, Any]]):
        return _dedup_rules(self.extract_penal_like_rules(articles))


class CriminalConstitutionExtractor(BaseLawExtractor, TopicsMixin):
    """Criminal Constitution extractor."""
    
    TOPIC_MAP = {
        ("لا جريمة ولا عقوبة الا", "لا جريمة ولا عقوبة إلا", "مبدأ الشرعية"): "Legality",
        ("قرينة البراءة", "البراءة", "متهم بريء", "المتهم بريء"): "PresumptionOfInnocence",
        ("حق الدفاع", "الحق في الدفاع", "حقوق الدفاع"): "DefenseRights",
        ("عدم رجعية", "لا رجعية", "عدم الرجعية"): "NonRetroactivity",
        ("التعذيب", "تعذيب", "حظر التعذيب"): "AntiTorture",
        ("الحرية الشخصية", "الحريه الشخصيه", "الحرية"): "PersonalLiberty",
        ("تفتيش", "التفتيش", "ضمانات التفتيش"): "SearchGuarantees",
        ("قبض", "القبض", "ضمانات القبض"): "ArrestGuarantees",
        ("محاكمة عادلة", "المحاكمة العادلة", "حق المحاكمة"): "FairTrial",
    }

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="criminal_constitution", law_name="الدستور الجنائي")

    def extract_topics(self, articles: List[Dict[str, Any]]):
        return self.extract_topics_by_map(articles)


class DrugsLawExtractor(BaseLawExtractor, PenalRulesMixin, TopicsMixin):
    """Anti-Drugs Law extractor."""
    
    TOPIC_MAP = {
        ("مخدر", "مخدرات", "المخدرات", "مواد مخدرة"): "Drugs",
        ("مؤثر عقلي", "مؤثرات عقلية", "المؤثرات العقلية"): "Psychotropics",
        ("جدول", "جداول", "الجدول", "الجداول"): "Schedules",
        ("تعاطي", "تعاطي المخدرات", "التعاطي"): "Use",
        ("إحراز", "احراز", "الإحراز"): "Possession",
        ("حيازة", "الحيازة"): "Possession",
        ("اتجار", "الاتجار", "اتجار بالمخدرات"): "Trafficking",
        ("توزيع", "التوزيع"): "Distribution",
        ("ترويج", "الترويج"): "Distribution",
        ("جلب", "استيراد", "الاستيراد"): "Importation",
        ("تصدير", "التصدير"): "Exportation",
        ("زراعة", "الزراعة"): "Cultivation",
        ("تصنيع", "التصنيع", "الإنتاج"): "Manufacturing",
        ("مصادرة", "المصادرة"): "Confiscation",
        ("يعاقب", "العقوبة"): "Penalty",
    }

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="anti_drugs", law_name="قانون مكافحة المخدرات")

    def extract_topics(self, articles: List[Dict[str, Any]]):
        return self.extract_topics_by_map(articles)

    def extract_rules(self, articles: List[Dict[str, Any]]):
        return _dedup_rules(self.extract_penal_like_rules(articles))


class CriminalProcedureExtractor(BaseLawExtractor, ProcedureRulesMixin, TopicsMixin):
    """Criminal Procedure Law extractor."""
    
    TOPIC_MAP = {
        ("قبض", "القبض", "أمر القبض"): "Arrest",
        ("تفتيش", "التفتيش", "أمر التفتيش"): "Search",
        ("حبس احتياطي", "الاحتياطي", "الحبس الاحتياطي"): "PretrialDetention",
        ("تحقيق", "التحقيق", "سلطة التحقيق"): "Investigation",
        ("استجواب", "الاستجواب"): "Interrogation",
        ("طعن", "الطعن", "طرق الطعن"): "Appeal",
        ("استئناف", "الاستئناف"): "Appeal",
        ("نقض", "النقض", "الطعن بالنقض"): "Cassation",
        ("بطلان", "البطلان"): "Nullity",
        ("ميعاد", "مواعيد", "الميعاد", "المواعيد"): "Deadlines",
        ("دليل", "أدلة", "الأدلة", "الإثبات"): "Evidence",
        ("محاكمة", "المحاكمة", "الجلسة"): "Trial",
    }

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="criminal_procedure", law_name="قانون الإجراءات الجنائية")

    def extract_topics(self, articles: List[Dict[str, Any]]):
        return self.extract_topics_by_map(articles)

    def extract_rules(self, articles: List[Dict[str, Any]]):
        return _dedup_rules(self.extract_procedure_rules(articles))


class AntiTerrorExtractor(BaseLawExtractor, PenalRulesMixin, TopicsMixin):
    """Anti-Terror Law extractor."""
    
    TOPIC_MAP = {
        ("إرهاب", "الارهاب", "الإرهاب", "إرهابي", "إرهابية"): "Terrorism",
        ("عمل إرهابي", "عمل ارهابي", "أعمال إرهابية"): "TerroristAct",
        ("تمويل", "تمويل الارهاب", "تمويل الإرهاب", "التمويل"): "Financing",
        ("تحريض", "التحريض", "التحريض على الإرهاب"): "Incitement",
        ("جماعة إرهابية", "منظمة إرهابية", "تنظيم إرهابي"): "TerroristOrganization",
        ("تجميد", "التجميد", "تجميد الأموال"): "Freezing",
        ("إدراج", "ادراج", "الإدراج", "القوائم"): "Listing",
        ("مصادرة", "المصادرة"): "Confiscation",
        ("يعاقب", "العقوبة"): "Penalty",
    }

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="anti_terror", law_name="قانون مكافحة الإرهاب")

    def extract_topics(self, articles: List[Dict[str, Any]]):
        return self.extract_topics_by_map(articles)

    def extract_rules(self, articles: List[Dict[str, Any]]):
        return _dedup_rules(self.extract_penal_like_rules(articles))


class EmergencyLawExtractor(BaseLawExtractor, TopicsMixin):
    """Emergency Law extractor."""
    
    TOPIC_MAP = {
        ("حالة الطوارئ", "حالة الطوارئ العامة", "الطوارئ"): "Emergency",
        ("إعلان الطوارئ", "اعلان الطوارئ"): "Declaration",
        ("تدابير", "اجراءات", "الإجراءات", "التدابير"): "Measures",
        ("تفتيش", "التفتيش"): "Search",
        ("قبض", "القبض"): "Arrest",
        ("حظر", "منع", "الحظر", "المنع"): "Restrictions",
        ("حظر التجول", "منع التجول"): "Curfew",
        ("مصادرة", "المصادرة"): "Confiscation",
        ("إغلاق", "غلق", "اغلاق", "الإغلاق"): "Closure",
        ("سلطات", "السلطات", "صلاحيات"): "Powers",
    }

    MEASURE_LINE_RE = re.compile(
        r"""(?ix)\b(?:يجوز(?:\s+ل)?|للرئيس|للسلط(?:ة|ات)|للجهات|تُ?منح\s+(?:السلطات|الصلاحيات)|يُ?خوّ?ل)\b.+""",
        re.VERBOSE
    )

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="emergency_law", law_name="قانون الطوارئ")

    def extract_topics(self, articles: List[Dict[str, Any]]):
        return self.extract_topics_by_map(articles)

    def extract_rules(self, articles: List[Dict[str, Any]]):
        rules = []
        for a in articles:
            text = a.get("text", "") or ""
            for line in text.splitlines():
                s = line.strip()
                if not s or len(s) < 15:
                    continue
                if self.MEASURE_LINE_RE.match(s):
                    rules.append({
                        "type": "EmergencyMeasureRule",
                        "article_no": a.get("article_no"),
                        "text": s
                    })
        return _dedup_rules(rules)


class CybercrimeExtractor(BaseLawExtractor, PenalRulesMixin, ProcedureRulesMixin, TopicsMixin):
    """Cybercrime Law extractor."""
    
    TOPIC_MAP = {
        ("نظام معلوماتي", "نظام المعلومات", "الأنظمة المعلوماتية"): "Cybercrime",
        ("شبكة معلوماتية", "شبكة المعلومات", "الشبكة المعلوماتية"): "Cybercrime",
        ("تقنية المعلومات", "تقنيه المعلومات"): "IT",
        ("موقع", "مواقع", "المواقع", "موقع إلكتروني"): "Web",
        ("حساب", "حسابات", "الحسابات"): "Account",
        ("بيانات", "البيانات", "معلومات"): "Data",
        ("قاعدة بيانات", "قواعد البيانات"): "Database",
        ("دخول غير مشروع", "دخول غير مصرح", "الدخول غير المصرح"): "UnauthorizedAccess",
        ("اختراق", "اختراقات", "الاختراق"): "UnauthorizedAccess",
        ("اعتراض", "الاعتراض"): "Interception",
        ("حجب", "الحجب"): "Blocking",
        ("مصادرة", "المصادرة"): "Confiscation",
        ("تفتيش", "التفتيش"): "Procedure",
        ("ضبط", "الضبط"): "Procedure",
        ("يعاقب", "العقوبة"): "Penalty",
    }

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="cybercrime", law_name="قانون مكافحة جرائم تقنية المعلومات")

    def extract_topics(self, articles: List[Dict[str, Any]]):
        return self.extract_topics_by_map(articles)

    def extract_rules(self, articles: List[Dict[str, Any]]):
        rules: List[Dict[str, Any]] = []
        rules.extend(self.extract_penal_like_rules(articles))
        rules.extend(self.extract_procedure_rules(articles))
        return _dedup_rules(rules)


def get_extractor(law_key: str, raw_text: str) -> BaseLawExtractor:
    """Factory function to get the appropriate extractor."""
    mapping = {
        "penal_code": PenalCodeExtractor,
        "money_laundering": MoneyLaunderingExtractor,
        "weapons_ammunition": WeaponsAmmunitionExtractor,
        "criminal_constitution": CriminalConstitutionExtractor,
        "anti_drugs": DrugsLawExtractor,
        "criminal_procedure": CriminalProcedureExtractor,
        "anti_terror": AntiTerrorExtractor,
        "emergency_law": EmergencyLawExtractor,
        "cybercrime": CybercrimeExtractor,
    }
    
    if law_key not in mapping:
        raise KeyError(
            f"Unknown law_key: '{law_key}'. "
            f"Available keys: {sorted(mapping.keys())}"
        )
    
    cls = mapping[law_key]
    return cls(raw_text)

print("✓ Law extractors defined")

# %%
# =============================================================================
# SECTION 7: FILE READING AND INGESTION
# =============================================================================

def _safe_read_text_file(
    fp: Path,
    encodings: Sequence[str],
    max_size_mb: Optional[int] = 50,
) -> Optional[str]:
    """Try reading a text file using a list of encodings."""
    try:
        size_mb = fp.stat().st_size / (1024 * 1024)
    except Exception:
        size_mb = 0.0

    if max_size_mb is not None and size_mb > max_size_mb:
        logger.warning("Skipping large file %s (%.1f MB > %d MB)", fp, size_mb, max_size_mb)
        return None

    for enc in encodings:
        try:
            content = fp.read_text(encoding=enc)
            logger.debug("Successfully read %s with encoding %s", fp.name, enc)
            return content
        except UnicodeDecodeError:
            try:
                content = fp.read_text(encoding=enc, errors="replace")
                logger.warning("Read %s with encoding %s (with replacements)", fp.name, enc)
                return content
            except Exception:
                continue
        except Exception as e:
            logger.debug("Failed reading %s with encoding %s: %s", fp, enc, e)
            continue

    try:
        raw = fp.read_bytes()
        content = raw.decode("utf-8", errors="replace")
        logger.warning("Read %s with UTF-8 fallback (may have errors)", fp.name)
        return content
    except Exception as e:
        logger.error("Failed to read file %s: %s", fp, e)
        return None


def read_all_txt_in_folder(
    folder: Path,
    encoding_list: Optional[Sequence[str]] = None,
    recursive: bool = False,
    file_pattern: str = "*.txt",
    max_file_size_mb: Optional[int] = 50,
) -> str:
    """Read and concatenate all .txt files in a folder."""
    folder = Path(folder)
    if not folder.exists() or not folder.is_dir():
        raise FileNotFoundError(f"Folder not found or not a directory: {folder}")

    encodings = list(encoding_list) if encoding_list else [
        "utf-8", "utf-16", "cp1256", "cp1252", "iso-8859-6", "latin1"
    ]

    globber = folder.rglob if recursive else folder.glob
    txt_files = sorted([p for p in globber(file_pattern) if p.is_file()])

    if not txt_files:
        logger.warning("No files matching '%s' found in %s", file_pattern, folder)
        return ""

    logger.info("Found %d file(s) matching '%s' in %s", len(txt_files), file_pattern, folder)

    parts: List[str] = []
    failed_files = []
    
    for fp in txt_files:
        try:
            text = _safe_read_text_file(fp, encodings=encodings, max_size_mb=max_file_size_mb)
            if text:
                parts.append(text)
                logger.debug("Read %s (%d chars)", fp.name, len(text))
            else:
                logger.warning("No content extracted from %s", fp.name)
                failed_files.append(fp.name)
        except Exception as e:
            logger.exception("Error reading file %s: %s", fp, e)
            failed_files.append(fp.name)

    if failed_files:
        logger.warning("Failed to read %d file(s): %s", len(failed_files), ", ".join(failed_files))

    result = "\n\n".join(parts).strip()
    logger.info("Total text length: %d characters", len(result))
    return result


def ingest_dataset(
    root_dir: str,
    folder_to_law_key: Dict[str, str],
    *,
    recursive_read: bool = False,
    encoding_list: Optional[Sequence[str]] = None,
    max_file_size_mb: Optional[int] = 50,
    verbose: bool = True,
    validate: bool = True,
) -> List[Dict[str, object]]:
    """Ingest entire dataset and extract all laws."""
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"Root dir not found: {root}")

    summaries: List[Dict[str, object]] = []
    folders = sorted([p for p in root.iterdir() if p.is_dir()])
    logger.info("Found %d folder(s) in %s", len(folders), root)

    for folder in folders:
        folder_name = folder.name

        if folder_name not in folder_to_law_key:
            logger.info("[SKIP] No mapping for folder: %s", folder_name)
            continue

        law_key = folder_to_law_key[folder_name]
        logger.info("=" * 70)
        logger.info("Processing: %s -> %s", folder_name, law_key)
        logger.info("=" * 70)

        try:
            raw_text = read_all_txt_in_folder(
                folder,
                encoding_list=encoding_list,
                recursive=recursive_read,
                max_file_size_mb=max_file_size_mb,
            )
        except Exception as e:
            logger.exception("Failed reading folder %s: %s", folder_name, e)
            summaries.append({
                "folder": folder_name,
                "law_key": law_key,
                "law_id": None,
                "articles": 0,
                "terms": 0,
                "references": 0,
                "topics": 0,
                "rules": 0,
                "error": f"read_error: {e}",
                "warnings": []
            })
            continue

        if not raw_text:
            logger.warning("[SKIP] Empty folder: %s", folder_name)
            summaries.append({
                "folder": folder_name,
                "law_key": law_key,
                "law_id": None,
                "articles": 0,
                "terms": 0,
                "references": 0,
                "topics": 0,
                "rules": 0,
                "error": "empty_folder",
                "warnings": []
            })
            continue

        try:
            extractor = get_extractor(law_key, raw_text)
        except Exception as e:
            logger.exception("Failed to create extractor for %s (%s): %s", folder_name, law_key, e)
            summaries.append({
                "folder": folder_name,
                "law_key": law_key,
                "law_id": None,
                "articles": 0,
                "terms": 0,
                "references": 0,
                "topics": 0,
                "rules": 0,
                "error": f"extractor_init_error: {e}",
                "warnings": []
            })
            continue

        try:
            bundle = extractor.extract_all()
        except Exception as e:
            logger.error("Error extracting data for %s: %s", folder_name, e)
            logger.debug(traceback.format_exc())
            summaries.append({
                "folder": folder_name,
                "law_key": law_key,
                "law_id": None,
                "articles": 0,
                "terms": 0,
                "references": 0,
                "topics": 0,
                "rules": 0,
                "error": f"extraction_error: {e}",
                "warnings": []
            })
            continue

        def safe_len(obj):
            try:
                return len(obj) if obj is not None else 0
            except Exception:
                return 0

        law_id = bundle.law_meta.get("law_id") if bundle.law_meta else None
        n_articles = safe_len(bundle.articles)
        n_terms = safe_len(bundle.terms)
        n_refs = safe_len(bundle.references)
        n_topics = safe_len(bundle.topics)
        n_rules = safe_len(bundle.rules)
        warnings = bundle.law_meta.get("validation_warnings", []) if bundle.law_meta else []

        if validate:
            if n_articles == 0:
                warnings.append("No articles extracted")
            elif n_articles == 1 and bundle.articles[0].get("article_no") is None:
                warnings.append("Only preamble extracted, no numbered articles")
            
            if n_articles > 0:
                avg_len = sum(len(a.get("text", "")) for a in bundle.articles) / n_articles
                if avg_len < 50:
                    warnings.append(f"Suspiciously short articles (avg {avg_len:.0f} chars)")
                if avg_len > 5000:
                    warnings.append(f"Suspiciously long articles (avg {avg_len:.0f} chars)")

        summary = {
            "folder": folder_name,
            "law_key": law_key,
            "law_id": law_id,
            "articles": n_articles,
            "terms": n_terms,
            "references": n_refs,
            "topics": n_topics,
            "rules": n_rules,
            "error": None,
            "warnings": warnings
        }
        summaries.append(summary)

        if verbose:
            logger.info(
                "[OK] Ingested %s -> law_id=%s articles=%d terms=%d refs=%d topics=%d rules=%d",
                folder_name, law_id, n_articles, n_terms, n_refs, n_topics, n_rules
            )
            if warnings:
                logger.warning("Warnings for %s: %s", folder_name, "; ".join(warnings))

    logger.info("=" * 70)
    logger.info("INGESTION COMPLETE")
    logger.info("=" * 70)
    total = len(summaries)
    success = len([s for s in summaries if s["error"] is None])
    logger.info("Total folders: %d", total)
    logger.info("Successful: %d", success)
    logger.info("Failed: %d", total - success)

    return summaries

print("✓ Ingestion functions defined")

# %%
# =============================================================================
# SECTION 8: VALIDATION UTILITIES
# =============================================================================

class ExtractionValidator:
    """Validates extraction quality and identifies issues."""
    
    def __init__(self, bundle, raw_text: str):
        self.bundle = bundle
        self.raw_text = raw_text
        self.issues = []
        self.warnings = []
        self.stats = {}
    
    def validate_all(self) -> Dict[str, Any]:
        """Run all validation checks."""
        self.validate_articles()
        self.validate_article_numbering()
        self.validate_terms()
        self.validate_references()
        self.validate_topics()
        self.validate_rules()
        self.calculate_statistics()
        
        return {
            "issues": self.issues,
            "warnings": self.warnings,
            "stats": self.stats,
            "overall_score": self.calculate_score()
        }
    
    def validate_articles(self):
        """Validate article extraction."""
        articles = self.bundle.articles
        
        if not articles:
            self.issues.append("No articles extracted")
            return
        
        empty_articles = [
            a.get("article_no") for a in articles 
            if not a.get("text") or len(a.get("text", "").strip()) == 0
        ]
        if empty_articles:
            self.issues.append(f"Found {len(empty_articles)} empty articles: {empty_articles}")
        
        short_articles = [
            (a.get("article_no"), len(a.get("text", "")))
            for a in articles 
            if a.get("article_no") and len(a.get("text", "")) < 20
        ]
        if short_articles:
            self.warnings.append(
                f"Found {len(short_articles)} very short articles: "
                f"{[f'{no}({l}chars)' for no, l in short_articles]}"
            )
        
        long_articles = [
            (a.get("article_no"), len(a.get("text", "")))
            for a in articles 
            if len(a.get("text", "")) > 10000
        ]
        if long_articles:
            self.warnings.append(
                f"Found {len(long_articles)} very long articles (>10k chars): "
                f"{[f'{no}({l}chars)' for no, l in long_articles]}"
            )
        
        total_article_text = sum(len(a.get("text", "")) for a in articles)
        coverage = (total_article_text / len(self.raw_text)) * 100 if self.raw_text else 0
        
        if coverage < 80:
            self.warnings.append(f"Low coverage: only {coverage:.1f}% of original text in articles")
        elif coverage > 105:
            self.warnings.append(f"High coverage: {coverage:.1f}% (possible duplication?)")
        
        self.stats["coverage_percent"] = coverage
    
    def validate_article_numbering(self):
        """Validate article number sequence."""
        articles = self.bundle.articles
        numbered = [a for a in articles if a.get("article_no") and str(a.get("article_no")) != "None"]
        
        if not numbered:
            self.issues.append("No numbered articles found")
            return
        
        article_nos = [a.get("article_no") for a in numbered]
        duplicates = [no for no, count in Counter(article_nos).items() if count > 1]
        if duplicates:
            self.issues.append(f"Duplicate article numbers: {duplicates}")
        
        numeric_articles = []
        for no in article_nos:
            match = re.match(r'(\d+)', str(no))
            if match:
                numeric_articles.append(int(match.group(1)))
        
        if numeric_articles:
            numeric_articles.sort()
            min_num = numeric_articles[0]
            max_num = numeric_articles[-1]
            expected_count = max_num - min_num + 1
            actual_count = len(set(numeric_articles))
            
            if actual_count < expected_count:
                missing = set(range(min_num, max_num + 1)) - set(numeric_articles)
                if missing and len(missing) < 20:
                    self.warnings.append(f"Potential gaps in numbering: missing {sorted(missing)}")
            
            self.stats["article_range"] = f"{min_num}-{max_num}"
            self.stats["expected_articles"] = expected_count
            self.stats["actual_articles"] = len(numbered)
    
    def validate_terms(self):
        """Validate term definitions."""
        terms = self.bundle.terms
        
        if not terms:
            self.warnings.append("No term definitions extracted")
            return
        
        empty = [t for t in terms if not t.get("term") or not t.get("definition")]
        if empty:
            self.issues.append(f"Found {len(empty)} incomplete term definitions")
        
        short_defs = [t.get("term") for t in terms if len(t.get("definition", "")) < 10]
        if short_defs:
            self.warnings.append(f"Found {len(short_defs)} very short definitions: {short_defs[:5]}")
        
        term_names = [t.get("term") for t in terms]
        duplicates = [t for t, count in Counter(term_names).items() if count > 1]
        if duplicates:
            self.warnings.append(f"Duplicate terms: {duplicates}")
        
        self.stats["terms_count"] = len(terms)
    
    def validate_references(self):
        """Validate article references."""
        references = self.bundle.references
        articles = self.bundle.articles
        
        if not references:
            self.warnings.append("No references extracted")
            return
        
        valid_nos = set(
            str(a.get("article_no")) 
            for a in articles 
            if a.get("article_no") and str(a.get("article_no")) != "None"
        )
        
        self_refs = [r for r in references if r.get("from_article_no") == r.get("to_article_no")]
        if self_refs:
            self.warnings.append(f"Found {len(self_refs)} self-references")
        
        invalid_refs = []
        for r in references:
            from_no = str(r.get("from_article_no", ""))
            to_no = str(r.get("to_article_no", ""))
            
            if from_no not in valid_nos and from_no != "None":
                invalid_refs.append(f"from:{from_no}")
            if to_no not in valid_nos:
                invalid_refs.append(f"to:{to_no}")
        
        if invalid_refs:
            self.warnings.append(f"Found {len(set(invalid_refs))} references to missing articles")
        
        self.stats["references_count"] = len(references)
    
    def validate_topics(self):
        """Validate topic extraction."""
        topics = self.bundle.topics
        
        if not topics:
            self.warnings.append("No topics extracted")
            return
        
        topic_names = [t.get("topic") for t in topics]
        topic_counts = Counter(topic_names)
        
        rare_topics = [t for t, count in topic_counts.items() if count == 1]
        if len(rare_topics) > len(topic_counts) * 0.5:
            self.warnings.append(
                f"Many topics appear only once ({len(rare_topics)}/{len(topic_counts)})"
            )
        
        self.stats["topics_count"] = len(topics)
        self.stats["unique_topics"] = len(topic_counts)
        self.stats["most_common_topics"] = topic_counts.most_common(5)
    
    def validate_rules(self):
        """Validate rule extraction."""
        rules = self.bundle.rules
        
        if not rules:
            self.warnings.append("No rules extracted")
            return
        
        rule_types = [r.get("type") for r in rules]
        type_counts = Counter(rule_types)
        
        empty_rules = [r for r in rules if not r.get("text")]
        if empty_rules:
            self.issues.append(f"Found {len(empty_rules)} rules with no text")
        
        self.stats["rules_count"] = len(rules)
        self.stats["rule_types"] = dict(type_counts)
    
    def calculate_statistics(self):
        """Calculate overall statistics."""
        articles = self.bundle.articles
        
        if articles:
            lengths = [len(a.get("text", "")) for a in articles if a.get("article_no")]
            if lengths:
                self.stats["avg_article_length"] = sum(lengths) / len(lengths)
                self.stats["min_article_length"] = min(lengths)
                self.stats["max_article_length"] = max(lengths)
        
        n_articles = len([a for a in articles if a.get("article_no")])
        if n_articles > 0:
            self.stats["terms_per_article"] = len(self.bundle.terms) / n_articles
            self.stats["refs_per_article"] = len(self.bundle.references) / n_articles
            self.stats["topics_per_article"] = len(self.bundle.topics) / n_articles
            self.stats["rules_per_article"] = len(self.bundle.rules) / n_articles
    
    def calculate_score(self) -> float:
        """Calculate overall quality score (0-100)."""
        score = 100.0
        score -= len(self.issues) * 10
        score -= len(self.warnings) * 2
        
        coverage = self.stats.get("coverage_percent", 0)
        if 85 <= coverage <= 100:
            score += 5
        
        if self.bundle.terms:
            score += 2
        if self.bundle.references:
            score += 2
        if self.bundle.topics:
            score += 2
        if self.bundle.rules:
            score += 2
        
        return max(0, min(100, score))
    
    def print_report(self):
        """Print a formatted validation report."""
        print("\n" + "=" * 70)
        print("EXTRACTION VALIDATION REPORT")
        print("=" * 70)
        
        print(f"\nBasic Statistics:")
        print(f"  Articles: {len(self.bundle.articles)}")
        print(f"  Terms: {len(self.bundle.terms)}")
        print(f"  References: {len(self.bundle.references)}")
        print(f"  Topics: {len(self.bundle.topics)}")
        print(f"  Rules: {len(self.bundle.rules)}")
        
        if self.issues:
            print(f"\n❌ ISSUES ({len(self.issues)}):")
            for issue in self.issues:
                print(f"  • {issue}")
        else:
            print("\n✓ No critical issues found")
        
        if self.warnings:
            print(f"\n⚠️  WARNINGS ({len(self.warnings)}):")
            for warning in self.warnings:
                print(f"  • {warning}")
        else:
            print("\n✓ No warnings")
        
        score = self.calculate_score()
        print(f"\n📊 Overall Quality Score: {score:.1f}/100")
        if score >= 90:
            print("   Grade: Excellent ✨")
        elif score >= 75:
            print("   Grade: Good ✓")
        elif score >= 60:
            print("   Grade: Acceptable ⚠️")
        else:
            print("   Grade: Needs Improvement ❌")
        
        print("\n" + "=" * 70 + "\n")

print("✓ Validation utilities defined")

# %%
# =============================================================================
# SECTION 9: CONFIGURATION
# =============================================================================

# SET YOUR PATHS HERE
ROOT_DIR = "Datasets/Unstructured_Data"
OUTPUT_JSON = "extraction_summary.json"

# Mapping of folder names to law keys
FOLDER_TO_LAW_KEY = {
    "3okobat": "penal_code",
    "8asl_amwal": "money_laundering",
    "Asle7a": "weapons_ammunition",
    "dostor_gena2y": "criminal_constitution",
    "drugs": "anti_drugs",
    "egra2at_gena2ya": "criminal_procedure",
    "erhab": "anti_terror",
    "taware2": "emergency_law",
    "technology": "cybercrime",
}

print("✓ Configuration set")
print(f"   Root directory: {ROOT_DIR}")
print(f"   Laws to extract: {len(FOLDER_TO_LAW_KEY)}")

# %%
# =============================================================================
# SECTION 10: RUN FULL EXTRACTION
# =============================================================================

print("\n" + "=" * 70)
print("STARTING FULL DATASET EXTRACTION")
print("=" * 70 + "\n")

try:
    summaries = ingest_dataset(
        root_dir=ROOT_DIR,
        folder_to_law_key=FOLDER_TO_LAW_KEY,
        recursive_read=False,
        max_file_size_mb=50,
        verbose=True,
        validate=True
    )
    
    print(f"\n✅ Successfully processed {len(summaries)} folders\n")
    
except Exception as e:
    print(f"\n❌ Error during ingestion: {e}\n")
    logger.exception("Full error details:")
    summaries = []

# %%
# =============================================================================
# SECTION 11: DISPLAY RESULTS
# =============================================================================

if summaries:
    # Convert to DataFrame
    df = pd.DataFrame(summaries)
    
    print("\n" + "=" * 70)
    print("EXTRACTION SUMMARY")
    print("=" * 70 + "\n")
    display(df)
    
    # Save to JSON
    output_path = Path(OUTPUT_JSON)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(summaries, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print(f"\n✓ Wrote JSON summary to {output_path}")
    
    # Statistics
    print("\n" + "=" * 70)
    print("STATISTICS")
    print("=" * 70)
    
    total = len(summaries)
    success = len([s for s in summaries if s["error"] is None])
    
    print(f"\nTotal folders: {total}")
    print(f"Successful: {success}")
    print(f"Failed: {total - success}")
    print(f"\nTotal extracted:")
    print(f"  Articles: {sum(s.get('articles', 0) for s in summaries)}")
    print(f"  Terms: {sum(s.get('terms', 0) for s in summaries)}")
    print(f"  References: {sum(s.get('references', 0) for s in summaries)}")
    print(f"  Topics: {sum(s.get('topics', 0) for s in summaries)}")
    print(f"  Rules: {sum(s.get('rules', 0) for s in summaries)}")
    
else:
    print("\n⚠️  No summaries to display.")

# %%
# =============================================================================
# SECTION 12: DETAILED VALIDATION FOR SPECIFIC LAW
# =============================================================================

2026-02-15 00:05:33,474 - INFO - Found 10 folder(s) in Datasets/Unstructured_Data
2026-02-15 00:05:33,474 - __main__ - INFO - Found 10 folder(s) in Datasets/Unstructured_Data
2026-02-15 00:05:33,474 - INFO - ======================================================================
2026-02-15 00:05:33,474 - __main__ - INFO - ======================================================================
2026-02-15 00:05:33,475 - INFO - Processing: 3okobat -> penal_code
2026-02-15 00:05:33,475 - __main__ - INFO - Processing: 3okobat -> penal_code
2026-02-15 00:05:33,475 - INFO - ======================================================================
2026-02-15 00:05:33,475 - __main__ - INFO - ======================================================================
2026-02-15 00:05:33,476 - INFO - Found 3 file(s) matching '*.txt' in Datasets/Unstructured_Data/3okobat
2026-02-15 00:05:33,476 - __main__ - INFO - Found 3 file(s) matching '*.txt' in Datasets/Unstructured_Data/3okobat
2026-02-15 00:05:33,477

/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject
✓ Imports complete
✓ Helper functions defined
✓ Data structures defined
✓ Base extractor defined
✓ Mixins defined
✓ Law extractors defined
✓ Ingestion functions defined
✓ Validation utilities defined
✓ Configuration set
   Root directory: Datasets/Unstructured_Data
   Laws to extract: 9

STARTING FULL DATASET EXTRACTION



2026-02-15 00:05:33,670 - __main__ - INFO - Found 20 article markers
2026-02-15 00:05:33,671 - INFO - Total extracted: 21 articles
2026-02-15 00:05:33,671 - __main__ - INFO - Total extracted: 21 articles
2026-02-15 00:05:33,672 - INFO - Extracted 12 references
2026-02-15 00:05:33,672 - __main__ - INFO - Extracted 12 references
2026-02-15 00:05:33,672 - INFO - Extracted 0 term definitions
2026-02-15 00:05:33,672 - __main__ - INFO - Extracted 0 term definitions
2026-02-15 00:05:33,676 - INFO - Extracted 18 topic associations
2026-02-15 00:05:33,676 - __main__ - INFO - Extracted 18 topic associations
2026-02-15 00:05:33,681 - INFO - Extracted 9 penal rules
2026-02-15 00:05:33,681 - __main__ - INFO - Extracted 9 penal rules
2026-02-15 00:05:33,682 - INFO - Extraction complete: 21 articles, 0 terms, 12 refs, 18 topics, 9 rules
2026-02-15 00:05:33,682 - __main__ - INFO - Extraction complete: 21 articles, 0 terms, 12 refs, 18 topics, 9 rules
2026-02-15 00:05:33,683 - INFO - [OK] Ingested 8asl


✅ Successfully processed 9 folders


EXTRACTION SUMMARY



,folder,law_key,law_id,articles,terms,references,topics,rules,error,warnings
0,3okobat,penal_code,penal_code,541,1,104,389,720,None,"[Short article: 64, Short article: 65, Short a..."
1,8asl_amwal,money_laundering,money_laundering,21,0,12,18,9,None,[]
2,Asle7a,weapons_ammunition,weapons_ammunition,52,0,11,58,38,None,[]
3,dostor_gena2y,criminal_constitution,criminal_constitution,252,0,6,0,0,None,"[Short article: 2, Short article: 3, Short art..."
4,drugs,anti_drugs,anti_drugs,69,0,26,59,59,None,[]
5,egra2at_gena2ya,criminal_procedure,criminal_procedure,511,0,86,365,214,None,"[Short article: 19, Short article: 20, Short a..."
6,erhab,anti_terror,anti_terror,55,0,2,5,6,None,"[Short article: 4, Short article: 5, Short art..."
7,taware2,emergency_law,emergency_law,21,0,1,13,7,None,[]
8,technology,cybercrime,cybercrime,46,0,8,82,74,None,[]



✓ Wrote JSON summary to extraction_summary.json

STATISTICS

Total folders: 9
Successful: 9
Failed: 0

Total extracted:
  Articles: 1568
  Terms: 1
  References: 256
  Topics: 989
  Rules: 1127


In [9]:
# Choose a law to validate in detail
LAW_TO_VALIDATE = "penal_code"  # Change this to test different laws

print("\n" + "=" * 70)
print(f"DETAILED VALIDATION: {LAW_TO_VALIDATE}")
print("=" * 70 + "\n")

try:
    folder_names = [k for k, v in FOLDER_TO_LAW_KEY.items() if v == LAW_TO_VALIDATE]
    
    if not folder_names:
        print(f"❌ No folder mapped to law key: {LAW_TO_VALIDATE}")
    else:
        folder_name = folder_names[0]
        folder_path = Path(ROOT_DIR) / folder_name
        
        # Read raw text
        print(f"Reading from: {folder_path}")
        raw_text = read_all_txt_in_folder(folder_path)
        
        # Extract
        print(f"Extracting with: {LAW_TO_VALIDATE} extractor")
        extractor = get_extractor(LAW_TO_VALIDATE, raw_text)
        bundle = extractor.extract_all()
        
        # Validate
        print("\nRunning validation...")
        validator = ExtractionValidator(bundle, raw_text)
        validation_results = validator.validate_all()
        validator.print_report()
        
        # Show sample articles
        print("=" * 70)
        print("SAMPLE ARTICLES (first 10)")
        print("=" * 70 + "\n")
        
        for i, article in enumerate(bundle.articles[:10]):
            art_no = article.get("article_no", "N/A")
            text = article.get("text", "")
            
            print(f"[{i+1}] Article {art_no}")
            print(f"    Length: {len(text)} chars")
            print(f"    Preview: {text[:200]}...")
            print("-" * 70)
        
        # Show extracted components
        print("\n" + "=" * 70)
        print("EXTRACTED COMPONENTS SAMPLES")
        print("=" * 70)
        
        if bundle.terms:
            print("\n📖 TERM DEFINITIONS (first 5):")
            for i, term in enumerate(bundle.terms[:5]):
                print(f"\n[{i+1}] Term: {term.get('term')}")
                print(f"    Article: {term.get('article_no')}")
                print(f"    Definition: {term.get('definition')[:150]}...")
        
        if bundle.references:
            print("\n🔗 REFERENCES (first 10):")
            for i, ref in enumerate(bundle.references[:10]):
                print(f"  [{i+1}] Article {ref.get('from_article_no')} → Article {ref.get('to_article_no')}")
        
        if bundle.topics:
            print("\n🏷️  TOPICS (first 10):")
            for i, topic in enumerate(bundle.topics[:10]):
                print(f"  [{i+1}] {topic.get('topic')} → Article {topic.get('article_no')}")
        
        if bundle.rules:
            print("\n⚖️  RULES (first 5):")
            for i, rule in enumerate(bundle.rules[:5]):
                print(f"\n[{i+1}] Type: {rule.get('type')}")
                print(f"    Article: {rule.get('article_no')}")
                print(f"    Text: {rule.get('text')[:200]}...")
        
except Exception as e:
    print(f"\n❌ Error during detailed validation: {e}")
    logger.exception("Full error details:")

# %%
# =============================================================================
# SECTION 13: EXPORT RESULTS
# =============================================================================

if 'bundle' in locals():
    output_dir = Path("extraction_output")
    output_dir.mkdir(exist_ok=True)
    
    print("\n" + "=" * 70)
    print("EXPORTING RESULTS")
    print("=" * 70 + "\n")
    
    # Save articles
    articles_df = pd.DataFrame(bundle.articles)
    articles_path = output_dir / f"{LAW_TO_VALIDATE}_articles.csv"
    articles_df.to_csv(articles_path, index=False, encoding='utf-8-sig')
    print(f"✓ Saved articles to {articles_path}")
    
    # Save terms
    if bundle.terms:
        terms_df = pd.DataFrame(bundle.terms)
        terms_path = output_dir / f"{LAW_TO_VALIDATE}_terms.csv"
        terms_df.to_csv(terms_path, index=False, encoding='utf-8-sig')
        print(f"✓ Saved terms to {terms_path}")
    
    # Save references
    if bundle.references:
        refs_df = pd.DataFrame(bundle.references)
        refs_path = output_dir / f"{LAW_TO_VALIDATE}_references.csv"
        refs_df.to_csv(refs_path, index=False, encoding='utf-8-sig')
        print(f"✓ Saved references to {refs_path}")
    
    # Save topics
    if bundle.topics:
        topics_df = pd.DataFrame(bundle.topics)
        topics_path = output_dir / f"{LAW_TO_VALIDATE}_topics.csv"
        topics_df.to_csv(topics_path, index=False, encoding='utf-8-sig')
        print(f"✓ Saved topics to {topics_path}")
    
    # Save rules
    if bundle.rules:
        rules_df = pd.DataFrame(bundle.rules)
        rules_path = output_dir / f"{LAW_TO_VALIDATE}_rules.csv"
        rules_df.to_csv(rules_path, index=False, encoding='utf-8-sig')
        print(f"✓ Saved rules to {rules_path}")
    
    print(f"\n✓ All results saved to {output_dir}/\n")

# %%
print("\n" + "=" * 70)
print("✅ ALL PROCESSING COMPLETE!")
print("=" * 70)
print("""
Next steps:
1. Review the extraction summary table above
2. Check quality scores (target: >90)
3. Review validation reports for warnings
4. Export and analyze specific laws
5. Adjust patterns if needed and re-run

For troubleshooting, check the validation reports and logs above.
For ~100% accuracy, iterate on laws with scores <90.
""")

2026-02-15 00:07:59,065 - INFO - Found 3 file(s) matching '*.txt' in Datasets/Unstructured_Data/3okobat
2026-02-15 00:07:59,065 - __main__ - INFO - Found 3 file(s) matching '*.txt' in Datasets/Unstructured_Data/3okobat
2026-02-15 00:07:59,067 - INFO - Total text length: 169853 characters
2026-02-15 00:07:59,067 - __main__ - INFO - Total text length: 169853 characters
2026-02-15 00:07:59,067 - INFO - Starting extraction for قانون العقوبات
2026-02-15 00:07:59,067 - __main__ - INFO - Starting extraction for قانون العقوبات
2026-02-15 00:07:59,069 - INFO - Found 540 article markers
2026-02-15 00:07:59,069 - __main__ - INFO - Found 540 article markers
2026-02-15 00:07:59,069 - WARNING - Article 64 is very short (5 chars)
2026-02-15 00:07:59,069 - __main__ - WARNING - Article 64 is very short (5 chars)
2026-02-15 00:07:59,070 - WARNING - Article 65 is very short (5 chars)
2026-02-15 00:07:59,070 - __main__ - WARNING - Article 65 is very short (5 chars)
2026-02-15 00:07:59,070 - WARNING - Arti


DETAILED VALIDATION: penal_code

Reading from: Datasets/Unstructured_Data/3okobat
Extracting with: penal_code extractor

Running validation...

EXTRACTION VALIDATION REPORT

Basic Statistics:
  Articles: 541
  Terms: 1
  References: 104
  Topics: 389
  Rules: 720

❌ ISSUES (2):
  • Found 2 empty articles: ['225', '242 مكرر (أ)']
  • Duplicate article numbers: ['1', '2', '77', '78', '79', '80', '82', '84', '86 مكرر', '88 مكرر', '98', '98 (أ)', '102', '106 مكرر', '109 مكرر', '116 مكرر', '118 مكرر', '124', '162', '178 مكرر', '204 مكرر', '309 مكرر', '316 مكرر', '323 مكرر']

⚠️  WARNINGS (1):
  • Found 59 very short articles: ['64(5chars)', '65(5chars)', '66(5chars)', '67(6chars)', '68(5chars)', '69(5chars)', '70(6chars)', '71(5chars)', '72(5chars)', '73(5chars)', '78(5chars)', '78(5chars)', '78 مكرر(5chars)', '79(6chars)', '79 مكرر(6chars)', '80(6chars)', '80(6chars)', '80 مكرر(6chars)', '81 مكرر(6chars)', '83 مكرر(6chars)', '84(6chars)', '84 مكرر(6chars)', '102(5chars)', '109(6chars)', '

In [12]:
"""
===============================================================================
LEGAL KNOWLEDGE GRAPH BUILDER - NEO4J
===============================================================================

This module builds and populates a comprehensive knowledge graph from extracted
Arabic legal texts using Neo4j.

Graph Schema:
- Nodes: Law, Article, Term, Topic, Rule
- Relationships: CONTAINS, REFERENCES, DEFINES, HAS_TOPIC, HAS_RULE, BELONGS_TO

Author: Claude (Anthropic)
Date: February 2026
Version: 2.0

===============================================================================
"""

# %%
# =============================================================================
# IMPORTS
# =============================================================================

from neo4j import GraphDatabase
from typing import Dict, List, Any, Optional, Tuple
import logging
from datetime import datetime
import hashlib
import json
from src.Config.config import NEO4J_PASSWORD , NEO4J_USERNAME , NEO4J_URI
logger = logging.getLogger(__name__)

print("✓ Imports complete")

# %%
# =============================================================================
# KNOWLEDGE GRAPH SCHEMA
# =============================================================================

"""
GRAPH SCHEMA:

Nodes:
------
(:Law)
  - law_id: string (unique)
  - name: string
  - text_length: int
  - created_at: datetime
  - article_count: int

(:Article)
  - article_id: string (unique, hash of law_id + article_no)
  - article_no: string
  - text: string
  - text_length: int
  - law_id: string

(:Term)
  - term_id: string (unique, hash of term + law_id)
  - term: string
  - definition: string
  - law_id: string

(:Topic)
  - topic_name: string (unique)
  - category: string (optional)

(:Rule)
  - rule_id: string (unique, hash of type + article_id + text)
  - type: string (PenaltyRule, CrimeRule, MeasureRule, ProcedureRule, etc.)
  - text: string
  - law_id: string

Relationships:
-------------
(:Law)-[:CONTAINS]->(:Article)
  properties: position (order in law)

(:Article)-[:REFERENCES]->(:Article)
  properties: reference_type (direct, indirect)

(:Article)-[:DEFINES]->(:Term)
  properties: none

(:Article)-[:HAS_TOPIC]->(:Topic)
  properties: relevance_score (optional)

(:Article)-[:HAS_RULE]->(:Rule)
  properties: rule_type

(:Term)-[:BELONGS_TO]->(:Law)
  properties: none

(:Rule)-[:BELONGS_TO]->(:Article)
  properties: none

Indexes:
--------
- Law.law_id
- Article.article_id
- Article.article_no
- Term.term_id
- Topic.topic_name
- Rule.rule_id
"""

print("✓ Schema defined")

# %%
# =============================================================================
# KNOWLEDGE GRAPH BUILDER CLASS
# =============================================================================

class LegalKnowledgeGraph:
    """
    Builds and manages a knowledge graph for legal data using Neo4j.
    """
    
    def __init__(self, uri: str, user: str, password: str):
        """
        Initialize connection to Neo4j database.
        
        Args:
            uri: Neo4j connection URI (e.g., "bolt://localhost:7687")
            user: Database username
            password: Database password
        """
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        logger.info(f"Connected to Neo4j at {uri}")
        
        # Verify connection
        try:
            self.driver.verify_connectivity()
            logger.info("✓ Neo4j connection verified")
        except Exception as e:
            logger.error(f"Failed to connect to Neo4j: {e}")
            raise
    
    def close(self):
        """Close database connection."""
        self.driver.close()
        logger.info("Neo4j connection closed")
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()
    
    # =========================================================================
    # SCHEMA SETUP
    # =========================================================================
    
    def setup_schema(self, drop_existing: bool = False):
        """
        Set up database schema: constraints and indexes.
        
        Args:
            drop_existing: If True, drop all existing data first
        """
        with self.driver.session() as session:
            if drop_existing:
                logger.warning("Dropping all existing data...")
                session.run("MATCH (n) DETACH DELETE n")
                logger.info("✓ All data cleared")
            
            # Drop existing constraints (in case of schema changes)
            logger.info("Setting up constraints and indexes...")
            
            constraints = [
                # Unique constraints
                "CREATE CONSTRAINT law_id_unique IF NOT EXISTS FOR (l:Law) REQUIRE l.law_id IS UNIQUE",
                "CREATE CONSTRAINT article_id_unique IF NOT EXISTS FOR (a:Article) REQUIRE a.article_id IS UNIQUE",
                "CREATE CONSTRAINT term_id_unique IF NOT EXISTS FOR (t:Term) REQUIRE t.term_id IS UNIQUE",
                "CREATE CONSTRAINT topic_name_unique IF NOT EXISTS FOR (t:Topic) REQUIRE t.topic_name IS UNIQUE",
                "CREATE CONSTRAINT rule_id_unique IF NOT EXISTS FOR (r:Rule) REQUIRE r.rule_id IS UNIQUE",
            ]
            
            for constraint in constraints:
                try:
                    session.run(constraint)
                    logger.debug(f"Created: {constraint[:50]}...")
                except Exception as e:
                    logger.debug(f"Constraint might already exist: {e}")
            
            # Additional indexes for performance
            indexes = [
                "CREATE INDEX article_no_index IF NOT EXISTS FOR (a:Article) ON (a.article_no)",
                "CREATE INDEX law_id_index IF NOT EXISTS FOR (a:Article) ON (a.law_id)",
                "CREATE INDEX term_text_index IF NOT EXISTS FOR (t:Term) ON (t.term)",
                "CREATE INDEX rule_type_index IF NOT EXISTS FOR (r:Rule) ON (r.type)",
            ]
            
            for index in indexes:
                try:
                    session.run(index)
                    logger.debug(f"Created: {index[:50]}...")
                except Exception as e:
                    logger.debug(f"Index might already exist: {e}")
            
            logger.info("✓ Schema setup complete")
    
    # =========================================================================
    # HELPER FUNCTIONS
    # =========================================================================
    
    @staticmethod
    def _generate_id(*components) -> str:
        """Generate a stable ID from components."""
        combined = "|".join(str(c) for c in components)
        return hashlib.sha256(combined.encode("utf-8")).hexdigest()[:16]
    
    # =========================================================================
    # NODE CREATION
    # =========================================================================
    
    def create_law(self, law_meta: Dict[str, Any]) -> str:
        """
        Create a Law node.
        
        Args:
            law_meta: Dictionary with law metadata
            
        Returns:
            law_id
        """
        with self.driver.session() as session:
            query = """
            MERGE (l:Law {law_id: $law_id})
            SET l.name = $name,
                l.text_length = $text_length,
                l.created_at = datetime(),
                l.updated_at = datetime()
            RETURN l.law_id as law_id
            """
            
            result = session.run(
                query,
                law_id=law_meta.get("law_id"),
                name=law_meta.get("name"),
                text_length=law_meta.get("text_length", 0)
            )
            
            record = result.single()
            law_id = record["law_id"] if record else law_meta.get("law_id")
            logger.info(f"✓ Created Law: {law_id}")
            return law_id
    
    def create_article(
        self, 
        law_id: str, 
        article_no: Optional[str], 
        text: str,
        position: int
    ) -> str:
        """
        Create an Article node and link it to a Law.
        
        Args:
            law_id: ID of the parent law
            article_no: Article number (can be None for preamble)
            text: Article text
            position: Position in the law (0-indexed)
            
        Returns:
            article_id
        """
        article_id = self._generate_id(law_id, article_no or "preamble", position)
        
        with self.driver.session() as session:
            query = """
            MATCH (l:Law {law_id: $law_id})
            MERGE (a:Article {article_id: $article_id})
            SET a.article_no = $article_no,
                a.text = $text,
                a.text_length = $text_length,
                a.law_id = $law_id,
                a.updated_at = datetime()
            MERGE (l)-[r:CONTAINS]->(a)
            SET r.position = $position
            RETURN a.article_id as article_id
            """
            
            result = session.run(
                query,
                law_id=law_id,
                article_id=article_id,
                article_no=article_no,
                text=text,
                text_length=len(text),
                position=position
            )
            
            record = result.single()
            logger.debug(f"✓ Created Article: {article_no} (position {position})")
            return article_id
    
    def create_term(
        self,
        law_id: str,
        term: str,
        definition: str,
        article_no: Optional[str]
    ) -> str:
        """
        Create a Term node and link it to an Article and Law.
        
        Args:
            law_id: ID of the law
            term: Term name
            definition: Term definition
            article_no: Article where term is defined
            
        Returns:
            term_id
        """
        term_id = self._generate_id(law_id, term)
        
        with self.driver.session() as session:
            query = """
            MATCH (l:Law {law_id: $law_id})
            MERGE (t:Term {term_id: $term_id})
            SET t.term = $term,
                t.definition = $definition,
                t.law_id = $law_id,
                t.updated_at = datetime()
            MERGE (t)-[:BELONGS_TO]->(l)
            
            WITH t, l
            OPTIONAL MATCH (l)-[:CONTAINS]->(a:Article {article_no: $article_no})
            WHERE a.law_id = $law_id
            FOREACH (_ IN CASE WHEN a IS NOT NULL THEN [1] ELSE [] END |
                MERGE (a)-[:DEFINES]->(t)
            )
            RETURN t.term_id as term_id
            """
            
            result = session.run(
                query,
                law_id=law_id,
                term_id=term_id,
                term=term,
                definition=definition,
                article_no=article_no
            )
            
            record = result.single()
            logger.debug(f"✓ Created Term: {term[:30]}...")
            return term_id
    
    def create_topic(self, topic_name: str, category: Optional[str] = None) -> str:
        """
        Create or update a Topic node.
        
        Args:
            topic_name: Name of the topic
            category: Optional category
            
        Returns:
            topic_name (used as ID)
        """
        with self.driver.session() as session:
            query = """
            MERGE (t:Topic {topic_name: $topic_name})
            SET t.category = $category,
                t.updated_at = datetime()
            RETURN t.topic_name as topic_name
            """
            
            result = session.run(
                query,
                topic_name=topic_name,
                category=category
            )
            
            record = result.single()
            logger.debug(f"✓ Created Topic: {topic_name}")
            return topic_name
    
    def create_rule(
        self,
        law_id: str,
        article_no: Optional[str],
        rule_type: str,
        text: str
    ) -> str:
        """
        Create a Rule node and link it to an Article.
        
        Args:
            law_id: ID of the law
            article_no: Article containing the rule
            rule_type: Type of rule (PenaltyRule, CrimeRule, etc.)
            text: Rule text
            
        Returns:
            rule_id
        """
        rule_id = self._generate_id(law_id, article_no, rule_type, text[:100])
        
        with self.driver.session() as session:
            query = """
            MATCH (l:Law {law_id: $law_id})
            OPTIONAL MATCH (l)-[:CONTAINS]->(a:Article {article_no: $article_no})
            WHERE a.law_id = $law_id
            
            MERGE (r:Rule {rule_id: $rule_id})
            SET r.type = $rule_type,
                r.text = $text,
                r.law_id = $law_id,
                r.updated_at = datetime()
            
            FOREACH (_ IN CASE WHEN a IS NOT NULL THEN [1] ELSE [] END |
                MERGE (a)-[rel:HAS_RULE]->(r)
                SET rel.rule_type = $rule_type
            )
            
            RETURN r.rule_id as rule_id
            """
            
            result = session.run(
                query,
                law_id=law_id,
                rule_id=rule_id,
                article_no=article_no,
                rule_type=rule_type,
                text=text
            )
            
            record = result.single()
            logger.debug(f"✓ Created Rule: {rule_type}")
            return rule_id
    
    # =========================================================================
    # RELATIONSHIP CREATION
    # =========================================================================
    
    def create_article_reference(
        self,
        law_id: str,
        from_article_no: str,
        to_article_no: str
    ):
        """
        Create a REFERENCES relationship between articles.
        
        Args:
            law_id: ID of the law
            from_article_no: Source article number
            to_article_no: Target article number
        """
        with self.driver.session() as session:
            query = """
            MATCH (l:Law {law_id: $law_id})
            MATCH (l)-[:CONTAINS]->(from:Article {article_no: $from_article_no})
            MATCH (l)-[:CONTAINS]->(to:Article {article_no: $to_article_no})
            WHERE from.law_id = $law_id AND to.law_id = $law_id
            MERGE (from)-[r:REFERENCES]->(to)
            SET r.created_at = datetime()
            """
            
            session.run(
                query,
                law_id=law_id,
                from_article_no=from_article_no,
                to_article_no=to_article_no
            )
            
            logger.debug(f"✓ Created reference: {from_article_no} -> {to_article_no}")
    
    def link_article_to_topic(
        self,
        law_id: str,
        article_no: str,
        topic_name: str
    ):
        """
        Link an article to a topic.
        
        Args:
            law_id: ID of the law
            article_no: Article number
            topic_name: Topic name
        """
        with self.driver.session() as session:
            query = """
            MATCH (l:Law {law_id: $law_id})
            MATCH (l)-[:CONTAINS]->(a:Article {article_no: $article_no})
            MATCH (t:Topic {topic_name: $topic_name})
            WHERE a.law_id = $law_id
            MERGE (a)-[r:HAS_TOPIC]->(t)
            SET r.created_at = datetime()
            """
            
            session.run(
                query,
                law_id=law_id,
                article_no=article_no,
                topic_name=topic_name
            )
            
            logger.debug(f"✓ Linked article {article_no} to topic {topic_name}")
    
    # =========================================================================
    # BATCH IMPORT FROM EXTRACTION
    # =========================================================================
    
    def import_law(self, bundle, verbose: bool = True) -> Dict[str, int]:
        """
        Import a complete law from extraction bundle.
        
        Args:
            bundle: ExtractedLaw object from extraction
            verbose: If True, print progress
            
        Returns:
            Dictionary with counts of imported items
        """
        stats = {
            "laws": 0,
            "articles": 0,
            "terms": 0,
            "topics": 0,
            "rules": 0,
            "references": 0,
        }
        
        law_id = bundle.law_meta.get("law_id")
        law_name = bundle.law_meta.get("name")
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"IMPORTING: {law_name} ({law_id})")
            print(f"{'='*70}\n")
        
        # 1. Create Law node
        self.create_law(bundle.law_meta)
        stats["laws"] = 1
        if verbose:
            print(f"✓ Created law node")
        
        # 2. Create Article nodes
        article_map = {}  # Map article_no to article_id
        for i, article in enumerate(bundle.articles):
            article_no = article.get("article_no")
            text = article.get("text", "")
            
            article_id = self.create_article(law_id, article_no, text, i)
            if article_no:
                article_map[article_no] = article_id
            
            stats["articles"] += 1
        
        if verbose:
            print(f"✓ Created {stats['articles']} articles")
        
        # 3. Create Term nodes
        for term in bundle.terms:
            self.create_term(
                law_id=law_id,
                term=term.get("term"),
                definition=term.get("definition"),
                article_no=term.get("article_no")
            )
            stats["terms"] += 1
        
        if verbose:
            print(f"✓ Created {stats['terms']} terms")
        
        # 4. Create Topic nodes and links
        unique_topics = set(t.get("topic") for t in bundle.topics)
        for topic_name in unique_topics:
            self.create_topic(topic_name)
        
        for topic in bundle.topics:
            self.link_article_to_topic(
                law_id=law_id,
                article_no=topic.get("article_no"),
                topic_name=topic.get("topic")
            )
            stats["topics"] += 1
        
        if verbose:
            print(f"✓ Created {len(unique_topics)} unique topics ({stats['topics']} links)")
        
        # 5. Create Rule nodes
        for rule in bundle.rules:
            self.create_rule(
                law_id=law_id,
                article_no=rule.get("article_no"),
                rule_type=rule.get("type"),
                text=rule.get("text")
            )
            stats["rules"] += 1
        
        if verbose:
            print(f"✓ Created {stats['rules']} rules")
        
        # 6. Create Reference relationships
        for ref in bundle.references:
            self.create_article_reference(
                law_id=law_id,
                from_article_no=ref.get("from_article_no"),
                to_article_no=ref.get("to_article_no")
            )
            stats["references"] += 1
        
        if verbose:
            print(f"✓ Created {stats['references']} references")
        
        # Update law with article count
        with self.driver.session() as session:
            session.run(
                "MATCH (l:Law {law_id: $law_id}) SET l.article_count = $count",
                law_id=law_id,
                count=stats["articles"]
            )
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"✅ IMPORT COMPLETE: {law_name}")
            print(f"{'='*70}\n")
        
        return stats
    
    def import_all_laws(
        self,
        summaries: List[Dict[str, Any]],
        extraction_function,
        verbose: bool = True
    ) -> Dict[str, Any]:
        """
        Import all laws from extraction summaries.
        
        Args:
            summaries: List of extraction summaries from ingest_dataset
            extraction_function: Function that takes (law_key, folder) and returns bundle
            verbose: If True, print progress
            
        Returns:
            Dictionary with total statistics
        """
        total_stats = {
            "laws": 0,
            "articles": 0,
            "terms": 0,
            "topics": 0,
            "rules": 0,
            "references": 0,
            "errors": []
        }
        
        successful = [s for s in summaries if s.get("error") is None]
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"IMPORTING {len(successful)} LAWS TO KNOWLEDGE GRAPH")
            print(f"{'='*70}\n")
        
        for summary in successful:
            law_key = summary.get("law_key")
            folder = summary.get("folder")
            
            try:
                # Get the bundle using extraction function
                bundle = extraction_function(law_key, folder)
                
                # Import to graph
                stats = self.import_law(bundle, verbose=verbose)
                
                # Accumulate stats
                for key in ["laws", "articles", "terms", "topics", "rules", "references"]:
                    total_stats[key] += stats.get(key, 0)
                
            except Exception as e:
                logger.error(f"Failed to import {law_key}: {e}")
                total_stats["errors"].append({
                    "law_key": law_key,
                    "error": str(e)
                })
                if verbose:
                    print(f"❌ Error importing {law_key}: {e}\n")
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"✅ ALL IMPORTS COMPLETE")
            print(f"{'='*70}")
            print(f"Total imported:")
            print(f"  Laws: {total_stats['laws']}")
            print(f"  Articles: {total_stats['articles']}")
            print(f"  Terms: {total_stats['terms']}")
            print(f"  Topics: {total_stats['topics']}")
            print(f"  Rules: {total_stats['rules']}")
            print(f"  References: {total_stats['references']}")
            if total_stats['errors']:
                print(f"  Errors: {len(total_stats['errors'])}")
            print(f"{'='*70}\n")
        
        return total_stats
    
    # =========================================================================
    # STATISTICS
    # =========================================================================
    
    def get_statistics(self) -> Dict[str, Any]:
        """
        Get comprehensive statistics about the knowledge graph.
        
        Returns:
            Dictionary with statistics
        """
        with self.driver.session() as session:
            stats = {}
            
            # Node counts
            stats["node_counts"] = {}
            for label in ["Law", "Article", "Term", "Topic", "Rule"]:
                result = session.run(f"MATCH (n:{label}) RETURN count(n) as count")
                stats["node_counts"][label] = result.single()["count"]
            
            # Relationship counts
            stats["relationship_counts"] = {}
            for rel_type in ["CONTAINS", "REFERENCES", "DEFINES", "HAS_TOPIC", "HAS_RULE", "BELONGS_TO"]:
                result = session.run(f"MATCH ()-[r:{rel_type}]->() RETURN count(r) as count")
                stats["relationship_counts"][rel_type] = result.single()["count"]
            
            # Most referenced articles
            result = session.run("""
                MATCH (a:Article)<-[:REFERENCES]-(other)
                RETURN a.law_id as law_id, a.article_no as article_no, count(other) as ref_count
                ORDER BY ref_count DESC
                LIMIT 10
            """)
            stats["most_referenced_articles"] = [
                {
                    "law_id": r["law_id"],
                    "article_no": r["article_no"],
                    "reference_count": r["ref_count"]
                }
                for r in result
            ]
            
            # Most common topics
            result = session.run("""
                MATCH (t:Topic)<-[:HAS_TOPIC]-(a:Article)
                RETURN t.topic_name as topic, count(a) as article_count
                ORDER BY article_count DESC
                LIMIT 10
            """)
            stats["most_common_topics"] = [
                {"topic": r["topic"], "article_count": r["article_count"]}
                for r in result
            ]
            
            # Rule type distribution
            result = session.run("""
                MATCH (r:Rule)
                RETURN r.type as rule_type, count(r) as count
                ORDER BY count DESC
            """)
            stats["rule_type_distribution"] = [
                {"rule_type": r["rule_type"], "count": r["count"]}
                for r in result
            ]
            
            return stats
    
    def print_statistics(self):
        """Print formatted statistics."""
        stats = self.get_statistics()
        
        print("\n" + "="*70)
        print("KNOWLEDGE GRAPH STATISTICS")
        print("="*70 + "\n")
        
        print("Node Counts:")
        for label, count in stats["node_counts"].items():
            print(f"  {label}: {count}")
        
        print("\nRelationship Counts:")
        for rel_type, count in stats["relationship_counts"].items():
            print(f"  {rel_type}: {count}")
        
        print("\nMost Referenced Articles:")
        for item in stats["most_referenced_articles"][:5]:
            print(f"  {item['law_id']} - Article {item['article_no']}: {item['reference_count']} references")
        
        print("\nMost Common Topics:")
        for item in stats["most_common_topics"][:5]:
            print(f"  {item['topic']}: {item['article_count']} articles")
        
        print("\nRule Type Distribution:")
        for item in stats["rule_type_distribution"]:
            print(f"  {item['rule_type']}: {item['count']}")
        
        print("\n" + "="*70 + "\n")

print("✓ Knowledge graph class defined")

# %%
# =============================================================================
# QUERY FUNCTIONS
# =============================================================================

class KnowledgeGraphQueries:
    """Useful queries for the legal knowledge graph."""
    
    def __init__(self, graph: LegalKnowledgeGraph):
        self.graph = graph
    
    def find_article(self, law_id: str, article_no: str) -> Dict[str, Any]:
        """Find a specific article and its connections."""
        with self.graph.driver.session() as session:
            query = """
            MATCH (l:Law {law_id: $law_id})-[:CONTAINS]->(a:Article {article_no: $article_no})
            OPTIONAL MATCH (a)-[:DEFINES]->(t:Term)
            OPTIONAL MATCH (a)-[:HAS_TOPIC]->(topic:Topic)
            OPTIONAL MATCH (a)-[:HAS_RULE]->(r:Rule)
            OPTIONAL MATCH (a)-[:REFERENCES]->(ref:Article)
            RETURN a, 
                   collect(DISTINCT t) as terms,
                   collect(DISTINCT topic.topic_name) as topics,
                   collect(DISTINCT r) as rules,
                   collect(DISTINCT ref.article_no) as references
            """
            
            result = session.run(query, law_id=law_id, article_no=article_no)
            record = result.single()
            
            if not record:
                return None
            
            return {
                "article": dict(record["a"]),
                "terms": [dict(t) for t in record["terms"]],
                "topics": record["topics"],
                "rules": [dict(r) for r in record["rules"]],
                "references": record["references"]
            }
    
    def search_articles_by_text(self, search_term: str, limit: int = 10) -> List[Dict]:
        """Search articles containing specific text."""
        with self.graph.driver.session() as session:
            query = """
            MATCH (l:Law)-[:CONTAINS]->(a:Article)
            WHERE a.text CONTAINS $search_term
            RETURN l.name as law_name, a.article_no as article_no, 
                   a.text as text, a.text_length as text_length
            LIMIT $limit
            """
            
            result = session.run(query, search_term=search_term, limit=limit)
            return [dict(r) for r in result]
    
    def get_articles_by_topic(self, topic_name: str) -> List[Dict]:
        """Get all articles with a specific topic."""
        with self.graph.driver.session() as session:
            query = """
            MATCH (l:Law)-[:CONTAINS]->(a:Article)-[:HAS_TOPIC]->(t:Topic {topic_name: $topic_name})
            RETURN l.name as law_name, l.law_id as law_id,
                   a.article_no as article_no, a.text as text
            ORDER BY l.law_id, a.article_no
            """
            
            result = session.run(query, topic_name=topic_name)
            return [dict(r) for r in result]
    
    def get_penalty_rules(self, law_id: Optional[str] = None) -> List[Dict]:
        """Get all penalty rules, optionally filtered by law."""
        with self.graph.driver.session() as session:
            if law_id:
                query = """
                MATCH (l:Law {law_id: $law_id})-[:CONTAINS]->(a:Article)-[:HAS_RULE]->(r:Rule)
                WHERE r.type = 'PenaltyRule'
                RETURN l.name as law_name, a.article_no as article_no, r.text as rule_text
                ORDER BY a.article_no
                """
                result = session.run(query, law_id=law_id)
            else:
                query = """
                MATCH (l:Law)-[:CONTAINS]->(a:Article)-[:HAS_RULE]->(r:Rule)
                WHERE r.type = 'PenaltyRule'
                RETURN l.name as law_name, a.article_no as article_no, r.text as rule_text
                ORDER BY l.name, a.article_no
                """
                result = session.run(query)
            
            return [dict(r) for r in result]
    
    def find_related_articles(
        self,
        law_id: str,
        article_no: str,
        max_depth: int = 2
    ) -> List[Dict]:
        """Find articles related through references (up to max_depth hops)."""
        with self.graph.driver.session() as session:
            query = f"""
            MATCH (l:Law {{law_id: $law_id}})-[:CONTAINS]->(start:Article {{article_no: $article_no}})
            MATCH path = (start)-[:REFERENCES*1..{max_depth}]->(related:Article)
            RETURN DISTINCT related.article_no as article_no, 
                   length(path) as distance,
                   related.text as text
            ORDER BY distance, article_no
            """
            
            result = session.run(query, law_id=law_id, article_no=article_no)
            return [dict(r) for r in result]
    
    def get_term_definitions(self, law_id: Optional[str] = None) -> List[Dict]:
        """Get all term definitions, optionally filtered by law."""
        with self.graph.driver.session() as session:
            if law_id:
                query = """
                MATCH (l:Law {law_id: $law_id})-[:CONTAINS]->(a:Article)-[:DEFINES]->(t:Term)
                RETURN l.name as law_name, a.article_no as article_no,
                       t.term as term, t.definition as definition
                ORDER BY t.term
                """
                result = session.run(query, law_id=law_id)
            else:
                query = """
                MATCH (l:Law)-[:CONTAINS]->(a:Article)-[:DEFINES]->(t:Term)
                RETURN l.name as law_name, a.article_no as article_no,
                       t.term as term, t.definition as definition
                ORDER BY l.name, t.term
                """
                result = session.run(query)
            
            return [dict(r) for r in result]
    
    def get_law_summary(self, law_id: str) -> Dict[str, Any]:
        """Get comprehensive summary of a law."""
        with self.graph.driver.session() as session:
            # Basic info
            result = session.run("""
                MATCH (l:Law {law_id: $law_id})
                OPTIONAL MATCH (l)-[:CONTAINS]->(a:Article)
                OPTIONAL MATCH (a)-[:DEFINES]->(t:Term)
                OPTIONAL MATCH (a)-[:HAS_RULE]->(r:Rule)
                RETURN l.name as name,
                       l.article_count as article_count,
                       count(DISTINCT a) as actual_articles,
                       count(DISTINCT t) as terms,
                       count(DISTINCT r) as rules
            """, law_id=law_id)
            
            summary = dict(result.single())
            
            # Topics
            result = session.run("""
                MATCH (l:Law {law_id: $law_id})-[:CONTAINS]->(a:Article)-[:HAS_TOPIC]->(t:Topic)
                RETURN t.topic_name as topic, count(a) as article_count
                ORDER BY article_count DESC
            """, law_id=law_id)
            
            summary["topics"] = [dict(r) for r in result]
            
            # Rule types
            result = session.run("""
                MATCH (l:Law {law_id: $law_id})-[:CONTAINS]->(a:Article)-[:HAS_RULE]->(r:Rule)
                RETURN r.type as rule_type, count(r) as count
                ORDER BY count DESC
            """, law_id=law_id)
            
            summary["rule_types"] = [dict(r) for r in result]
            
            return summary

print("✓ Query functions defined")

# %%
print("\n" + "="*70)
print("✅ KNOWLEDGE GRAPH MODULE LOADED")
print("="*70)
print("""
To use:
1. Set up Neo4j database
2. Initialize: graph = LegalKnowledgeGraph(uri, user, password)
3. Setup schema: graph.setup_schema()
4. Import laws: graph.import_law(bundle)
5. Query: queries = KnowledgeGraphQueries(graph)
""")

✓ Imports complete
✓ Schema defined
✓ Knowledge graph class defined
✓ Query functions defined

✅ KNOWLEDGE GRAPH MODULE LOADED

To use:
1. Set up Neo4j database
2. Initialize: graph = LegalKnowledgeGraph(uri, user, password)
3. Setup schema: graph.setup_schema()
4. Import laws: graph.import_law(bundle)
5. Query: queries = KnowledgeGraphQueries(graph)



In [14]:
"""
===============================================================================
COMPLETE LEGAL DATA PIPELINE: EXTRACTION → KNOWLEDGE GRAPH
===============================================================================

This notebook integrates the extraction system with the knowledge graph.
It extracts legal texts and builds a comprehensive Neo4j knowledge graph.

Workflow:
1. Extract data from text files
2. Validate extraction quality
3. Build knowledge graph in Neo4j
4. Query and analyze the graph

===============================================================================
"""

# %%
# =============================================================================
# SECTION 1: IMPORTS
# =============================================================================

# Import extraction system (from complete_extraction_system.py)
import sys
from pathlib import Path
import logging
import pandas as pd
from typing import Dict, List, Any

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ All imports complete")

# %%
# =============================================================================
# SECTION 2: CONFIGURATION
# =============================================================================

# Extraction Configuration
ROOT_DIR = "Datasets/Unstructured_Data"
FOLDER_TO_LAW_KEY = {
    "3okobat": "penal_code",
    "8asl_amwal": "money_laundering",
    "Asle7a": "weapons_ammunition",
    "dostor_gena2y": "criminal_constitution",
    "drugs": "anti_drugs",
    "egra2at_gena2ya": "criminal_procedure",
    "erhab": "anti_terror",
    "taware2": "emergency_law",
    "technology": "cybercrime",
}

# Neo4j Configuration
NEO4J_URI = NEO4J_URI
NEO4J_USER = NEO4J_USERNAME
NEO4J_PASSWORD = NEO4J_PASSWORD

# Options
DROP_EXISTING_DATA = True  # Set to True to clear database before import
VALIDATE_EXTRACTION = True  # Set to True to validate each extraction

print("✓ Configuration set")
print(f"   Data directory: {ROOT_DIR}")
print(f"   Neo4j URI: {NEO4J_URI}")
print(f"   Neo4j User: {NEO4J_USER}")
print(f"   Laws to process: {len(FOLDER_TO_LAW_KEY)}")

# %%
# =============================================================================
# SECTION 3: EXTRACT ALL LAWS
# =============================================================================

print("\n" + "="*70)
print("STEP 1: EXTRACTING ALL LAWS")
print("="*70 + "\n")

summaries = ingest_dataset(
    root_dir=ROOT_DIR,
    folder_to_law_key=FOLDER_TO_LAW_KEY,
    recursive_read=False,
    max_file_size_mb=50,
    verbose=True,
    validate=VALIDATE_EXTRACTION
)

# Display summary
df = pd.DataFrame(summaries)
print("\n" + "="*70)
print("EXTRACTION SUMMARY")
print("="*70 + "\n")
display(df)

successful_summaries = [s for s in summaries if s.get("error") is None]
print(f"\n✓ Successfully extracted {len(successful_summaries)}/{len(summaries)} laws\n")

# %%
# =============================================================================
# SECTION 4: INITIALIZE KNOWLEDGE GRAPH
# =============================================================================

print("\n" + "="*70)
print("STEP 2: CONNECTING TO NEO4J")
print("="*70 + "\n")

try:
    graph = LegalKnowledgeGraph(
        uri=NEO4J_URI,
        user=NEO4J_USER,
        password=NEO4J_PASSWORD
    )
    
    # Setup schema
    graph.setup_schema(drop_existing=DROP_EXISTING_DATA)
    
    print("✓ Connected to Neo4j")
    print("✓ Schema setup complete")
    
except Exception as e:
    print(f"❌ Failed to connect to Neo4j: {e}")
    print("\nPlease ensure:")
    print("  1. Neo4j is running")
    print("  2. NEO4J_PASSWORD is correct")
    print("  3. Connection details are correct")
    raise

# %%
# =============================================================================
# SECTION 5: IMPORT LAWS TO KNOWLEDGE GRAPH
# =============================================================================

print("\n" + "="*70)
print("STEP 3: BUILDING KNOWLEDGE GRAPH")
print("="*70 + "\n")

# Function to extract a specific law
def extract_law(law_key: str, folder_name: str) -> ExtractedLaw:
    """Extract a law given its key and folder name."""
    folder_path = Path(ROOT_DIR) / folder_name
    raw_text = read_all_txt_in_folder(folder_path)
    extractor = get_extractor(law_key, raw_text)
    bundle = extractor.extract_all()
    return bundle

# Import all laws
total_stats = {
    "laws": 0,
    "articles": 0,
    "terms": 0,
    "topics": 0,
    "rules": 0,
    "references": 0,
    "errors": []
}

for summary in successful_summaries:
    law_key = summary.get("law_key")
    folder = summary.get("folder")
    
    try:
        print(f"\n{'='*70}")
        print(f"Importing: {law_key}")
        print(f"{'='*70}\n")
        
        # Extract
        bundle = extract_law(law_key, folder)
        
        # Import to graph
        stats = graph.import_law(bundle, verbose=True)
        
        # Accumulate stats
        for key in ["laws", "articles", "terms", "topics", "rules", "references"]:
            total_stats[key] += stats.get(key, 0)
        
    except Exception as e:
        logger.error(f"Failed to import {law_key}: {e}")
        total_stats["errors"].append({"law_key": law_key, "error": str(e)})
        print(f"❌ Error: {e}\n")

# Display total statistics
print("\n" + "="*70)
print("✅ KNOWLEDGE GRAPH BUILD COMPLETE")
print("="*70)
print(f"\nTotal imported:")
print(f"  Laws: {total_stats['laws']}")
print(f"  Articles: {total_stats['articles']}")
print(f"  Terms: {total_stats['terms']}")
print(f"  Topics: {total_stats['topics']}")
print(f"  Rules: {total_stats['rules']}")
print(f"  References: {total_stats['references']}")

if total_stats['errors']:
    print(f"\n⚠️  Errors: {len(total_stats['errors'])}")
    for err in total_stats['errors']:
        print(f"    {err['law_key']}: {err['error']}")

print("\n" + "="*70 + "\n")

# %%
# =============================================================================
# SECTION 6: KNOWLEDGE GRAPH STATISTICS
# =============================================================================

print("\n" + "="*70)
print("STEP 4: ANALYZING KNOWLEDGE GRAPH")
print("="*70 + "\n")

graph.print_statistics()

# %%
# =============================================================================
# SECTION 7: EXAMPLE QUERIES
# =============================================================================

print("\n" + "="*70)
print("STEP 5: EXAMPLE QUERIES")
print("="*70 + "\n")

queries = KnowledgeGraphQueries(graph)

# Example 1: Search for articles about penalties
print("\n1️⃣ Articles containing 'يعاقب':")
print("-" * 70)
results = queries.search_articles_by_text("يعاقب", limit=5)
for i, r in enumerate(results, 1):
    print(f"\n[{i}] {r['law_name']} - Article {r['article_no']}")
    print(f"    Text: {r['text'][:150]}...")

# Example 2: Get articles by topic
print("\n\n2️⃣ Articles with topic 'Penalty':")
print("-" * 70)
results = queries.get_articles_by_topic("Penalty")
print(f"Found {len(results)} articles")
for i, r in enumerate(results[:5], 1):
    print(f"  [{i}] {r['law_name']} - Article {r['article_no']}")

# Example 3: Get penalty rules
print("\n\n3️⃣ Penalty Rules (sample):")
print("-" * 70)
results = queries.get_penalty_rules()
print(f"Found {len(results)} penalty rules")
for i, r in enumerate(results[:5], 1):
    print(f"\n[{i}] {r['law_name']} - Article {r['article_no']}")
    print(f"    Rule: {r['rule_text'][:200]}...")

# Example 4: Get term definitions
print("\n\n4️⃣ Term Definitions (sample):")
print("-" * 70)
results = queries.get_term_definitions()
print(f"Found {len(results)} terms")
for i, r in enumerate(results[:5], 1):
    print(f"\n[{i}] {r['law_name']} - Article {r['article_no']}")
    print(f"    Term: {r['term']}")
    print(f"    Definition: {r['definition'][:150]}...")

# Example 5: Law summary
print("\n\n5️⃣ Law Summary - Money Laundering:")
print("-" * 70)
summary = queries.get_law_summary("money_laundering")
print(f"\nLaw: {summary['name']}")
print(f"Articles: {summary['actual_articles']}")
print(f"Terms: {summary['terms']}")
print(f"Rules: {summary['rules']}")
print(f"\nTop Topics:")
for topic in summary['topics'][:5]:
    print(f"  • {topic['topic']}: {topic['article_count']} articles")
print(f"\nRule Types:")
for rt in summary['rule_types']:
    print(f"  • {rt['rule_type']}: {rt['count']}")

# %%
# =============================================================================
# SECTION 8: CUSTOM QUERIES
# =============================================================================

print("\n\n" + "="*70)
print("ADVANCED CUSTOM QUERIES")
print("="*70 + "\n")

# Query 1: Find all articles that reference Article 1
print("6️⃣ Articles referencing Article '1' in any law:")
print("-" * 70)
with graph.driver.session() as session:
    result = session.run("""
        MATCH (l:Law)-[:CONTAINS]->(from:Article)-[:REFERENCES]->(to:Article {article_no: '1'})
        RETURN l.name as law_name, from.article_no as from_article, to.article_no as to_article
        LIMIT 10
    """)
    
    refs = list(result)
    print(f"Found {len(refs)} references")
    for i, r in enumerate(refs, 1):
        print(f"  [{i}] {r['law_name']}: Article {r['from_article']} → Article {r['to_article']}")

# Query 2: Most connected articles (hub articles)
print("\n\n7️⃣ Most referenced articles (hubs):")
print("-" * 70)
with graph.driver.session() as session:
    result = session.run("""
        MATCH (l:Law)-[:CONTAINS]->(a:Article)
        OPTIONAL MATCH (a)<-[:REFERENCES]-(other)
        WITH l, a, count(other) as incoming
        WHERE incoming > 0
        RETURN l.name as law_name, a.article_no as article_no, incoming
        ORDER BY incoming DESC
        LIMIT 10
    """)
    
    for i, r in enumerate(result, 1):
        print(f"  [{i}] {r['law_name']} - Article {r['article_no']}: {r['incoming']} incoming references")

# Query 3: Articles with multiple topics
print("\n\n8️⃣ Articles with multiple topics:")
print("-" * 70)
with graph.driver.session() as session:
    result = session.run("""
        MATCH (l:Law)-[:CONTAINS]->(a:Article)-[:HAS_TOPIC]->(t:Topic)
        WITH l, a, collect(t.topic_name) as topics
        WHERE size(topics) > 2
        RETURN l.name as law_name, a.article_no as article_no, topics, size(topics) as topic_count
        ORDER BY topic_count DESC
        LIMIT 10
    """)
    
    for i, r in enumerate(result, 1):
        print(f"  [{i}] {r['law_name']} - Article {r['article_no']}: {r['topic_count']} topics")
        print(f"      Topics: {', '.join(r['topics'][:5])}")

# Query 4: Find chains of references
print("\n\n9️⃣ Reference chains (Article → Article → Article):")
print("-" * 70)
with graph.driver.session() as session:
    result = session.run("""
        MATCH path = (a1:Article)-[:REFERENCES]->(a2:Article)-[:REFERENCES]->(a3:Article)
        WHERE a1.law_id = a2.law_id AND a2.law_id = a3.law_id
        RETURN a1.law_id as law_id, 
               a1.article_no as art1, 
               a2.article_no as art2, 
               a3.article_no as art3
        LIMIT 5
    """)
    
    chains = list(result)
    if chains:
        print(f"Found {len(chains)} reference chains")
        for i, r in enumerate(chains, 1):
            print(f"  [{i}] {r['law_id']}: Article {r['art1']} → {r['art2']} → {r['art3']}")
    else:
        print("  No reference chains found")

# Query 10: Topic co-occurrence
print("\n\n🔟 Topic co-occurrence (topics that appear together):")
print("-" * 70)
with graph.driver.session() as session:
    result = session.run("""
        MATCH (a:Article)-[:HAS_TOPIC]->(t1:Topic)
        MATCH (a)-[:HAS_TOPIC]->(t2:Topic)
        WHERE t1.topic_name < t2.topic_name
        RETURN t1.topic_name as topic1, t2.topic_name as topic2, count(a) as co_occurrence
        ORDER BY co_occurrence DESC
        LIMIT 10
    """)
    
    for i, r in enumerate(result, 1):
        print(f"  [{i}] {r['topic1']} + {r['topic2']}: {r['co_occurrence']} articles")

# %%
# =============================================================================
# SECTION 9: EXPORT QUERY RESULTS
# =============================================================================

print("\n\n" + "="*70)
print("EXPORTING QUERY RESULTS")
print("="*70 + "\n")

output_dir = Path("graph_exports")
output_dir.mkdir(exist_ok=True)

# Export 1: All penalty rules
print("Exporting penalty rules...")
results = queries.get_penalty_rules()
df = pd.DataFrame(results)
df.to_csv(output_dir / "penalty_rules.csv", index=False, encoding='utf-8-sig')
print(f"✓ Saved {len(results)} penalty rules to penalty_rules.csv")

# Export 2: All term definitions
print("Exporting term definitions...")
results = queries.get_term_definitions()
df = pd.DataFrame(results)
df.to_csv(output_dir / "term_definitions.csv", index=False, encoding='utf-8-sig')
print(f"✓ Saved {len(results)} term definitions to term_definitions.csv")

# Export 3: Topic distribution
print("Exporting topic distribution...")
stats = graph.get_statistics()
df = pd.DataFrame(stats['most_common_topics'])
df.to_csv(output_dir / "topic_distribution.csv", index=False, encoding='utf-8-sig')
print(f"✓ Saved topic distribution to topic_distribution.csv")

# Export 4: Most referenced articles
print("Exporting most referenced articles...")
df = pd.DataFrame(stats['most_referenced_articles'])
df.to_csv(output_dir / "most_referenced_articles.csv", index=False, encoding='utf-8-sig')
print(f"✓ Saved most referenced articles to most_referenced_articles.csv")

print(f"\n✓ All exports saved to {output_dir}/\n")

# %%
# =============================================================================
# SECTION 10: CLEANUP
# =============================================================================

print("\n" + "="*70)
print("✅ PIPELINE COMPLETE!")
print("="*70)
print("""
Summary:
1. ✓ Extracted all laws from text files
2. ✓ Built comprehensive knowledge graph
3. ✓ Populated Neo4j database
4. ✓ Ran example queries
5. ✓ Exported results

Next steps:
- Open Neo4j Browser to visualize the graph
- Run custom Cypher queries
- Build applications using the knowledge graph
- Analyze legal relationships and patterns

Connection info:
- URI: {uri}
- User: {user}
- Database: neo4j

To close the connection: graph.close()
""".format(uri=NEO4J_URI, user=NEO4J_USER))

# Note: Uncomment to close connection
# graph.close()

2026-02-15 00:15:25,325 - INFO - Found 10 folder(s) in Datasets/Unstructured_Data
2026-02-15 00:15:25,325 - __main__ - INFO - Found 10 folder(s) in Datasets/Unstructured_Data
2026-02-15 00:15:25,326 - INFO - ======================================================================
2026-02-15 00:15:25,326 - __main__ - INFO - ======================================================================
2026-02-15 00:15:25,326 - INFO - Processing: 3okobat -> penal_code
2026-02-15 00:15:25,326 - __main__ - INFO - Processing: 3okobat -> penal_code
2026-02-15 00:15:25,327 - INFO - ======================================================================
2026-02-15 00:15:25,327 - __main__ - INFO - ======================================================================
2026-02-15 00:15:25,328 - INFO - Found 3 file(s) matching '*.txt' in Datasets/Unstructured_Data/3okobat
2026-02-15 00:15:25,328 - __main__ - INFO - Found 3 file(s) matching '*.txt' in Datasets/Unstructured_Data/3okobat
2026-02-15 00:15:25,329

✓ All imports complete
✓ Configuration set
   Data directory: Datasets/Unstructured_Data
   Neo4j URI: neo4j+s://8e3595a2.databases.neo4j.io
   Neo4j User: neo4j
   Laws to process: 9

STEP 1: EXTRACTING ALL LAWS



2026-02-15 00:15:25,527 - __main__ - INFO - Extracted 58 topic associations
2026-02-15 00:15:25,532 - INFO - Extracted 38 penal rules
2026-02-15 00:15:25,532 - __main__ - INFO - Extracted 38 penal rules
2026-02-15 00:15:25,532 - INFO - Extraction complete: 52 articles, 0 terms, 11 refs, 58 topics, 38 rules
2026-02-15 00:15:25,532 - __main__ - INFO - Extraction complete: 52 articles, 0 terms, 11 refs, 58 topics, 38 rules
2026-02-15 00:15:25,532 - INFO - [OK] Ingested Asle7a -> law_id=weapons_ammunition articles=52 terms=0 refs=11 topics=58 rules=38
2026-02-15 00:15:25,532 - __main__ - INFO - [OK] Ingested Asle7a -> law_id=weapons_ammunition articles=52 terms=0 refs=11 topics=58 rules=38
2026-02-15 00:15:25,533 - INFO - ======================================================================
2026-02-15 00:15:25,533 - __main__ - INFO - ======================================================================
2026-02-15 00:15:25,533 - INFO - Processing: dostor_gena2y -> criminal_constitution
20


EXTRACTION SUMMARY



,folder,law_key,law_id,articles,terms,references,topics,rules,error,warnings
0,3okobat,penal_code,penal_code,541,1,104,389,720,None,"[Short article: 64, Short article: 65, Short a..."
1,8asl_amwal,money_laundering,money_laundering,21,0,12,18,9,None,[]
2,Asle7a,weapons_ammunition,weapons_ammunition,52,0,11,58,38,None,[]
3,dostor_gena2y,criminal_constitution,criminal_constitution,252,0,6,0,0,None,"[Short article: 2, Short article: 3, Short art..."
4,drugs,anti_drugs,anti_drugs,69,0,26,59,59,None,[]
5,egra2at_gena2ya,criminal_procedure,criminal_procedure,511,0,86,365,214,None,"[Short article: 19, Short article: 20, Short a..."
6,erhab,anti_terror,anti_terror,55,0,2,5,6,None,"[Short article: 4, Short article: 5, Short art..."
7,taware2,emergency_law,emergency_law,21,0,1,13,7,None,[]
8,technology,cybercrime,cybercrime,46,0,8,82,74,None,[]


...


✅ IMPORT COMPLETE: قانون مكافحة الإرهاب


Importing: emergency_law


IMPORTING: قانون الطوارئ (emergency_law)

✓ Created law node
✓ Created 21 articles
✓ Created 0 terms
✓ Created 5 unique topics (13 links)
✓ Created 7 rules


2026-02-15 00:33:38,940 - INFO - Found 1 file(s) matching '*.txt' in Datasets/Unstructured_Data/technology
2026-02-15 00:33:38,940 - __main__ - INFO - Found 1 file(s) matching '*.txt' in Datasets/Unstructured_Data/technology
2026-02-15 00:33:38,941 - INFO - Total text length: 30544 characters
2026-02-15 00:33:38,941 - __main__ - INFO - Total text length: 30544 characters
2026-02-15 00:33:38,941 - INFO - Starting extraction for قانون مكافحة جرائم تقنية المعلومات
2026-02-15 00:33:38,941 - __main__ - INFO - Starting extraction for قانون مكافحة جرائم تقنية المعلومات
2026-02-15 00:33:38,942 - INFO - Found 45 article markers
2026-02-15 00:33:38,942 - __main__ - INFO - Found 45 article markers
2026-02-15 00:33:38,942 - INFO - Total extracted: 46 articles
2026-02-15 00:33:38,942 - __main__ - INFO - Total extracted: 46 articles
2026-02-15 00:33:38,943 - INFO - Extracted 8 references
2026-02-15 00:33:38,943 - __main__ - INFO - Extracted 8 references
2026-02-15 00:33:38,944 - INFO - Extracted 0 t

✓ Created 1 references

✅ IMPORT COMPLETE: قانون الطوارئ


Importing: cybercrime



2026-02-15 00:33:39,951 - INFO - Extracted 10 procedure rules
2026-02-15 00:33:39,951 - __main__ - INFO - Extracted 10 procedure rules
2026-02-15 00:33:39,952 - INFO - Extraction complete: 46 articles, 0 terms, 8 refs, 82 topics, 74 rules
2026-02-15 00:33:39,952 - __main__ - INFO - Extraction complete: 46 articles, 0 terms, 8 refs, 82 topics, 74 rules
2026-02-15 00:33:40,093 - INFO - ✓ Created Law: cybercrime
2026-02-15 00:33:40,093 - __main__ - INFO - ✓ Created Law: cybercrime



IMPORTING: قانون مكافحة جرائم تقنية المعلومات (cybercrime)

✓ Created law node
✓ Created 46 articles
✓ Created 0 terms
✓ Created 10 unique topics (82 links)
✓ Created 74 rules
✓ Created 8 references

✅ IMPORT COMPLETE: قانون مكافحة جرائم تقنية المعلومات


✅ KNOWLEDGE GRAPH BUILD COMPLETE

Total imported:
  Laws: 9
  Articles: 1568
  Terms: 1
  Topics: 989
  Rules: 1127
  References: 256



STEP 4: ANALYZING KNOWLEDGE GRAPH


KNOWLEDGE GRAPH STATISTICS

Node Counts:
  Law: 9
  Article: 1568
  Term: 1
  Topic: 44
  Rule: 1126

Relationship Counts:
  CONTAINS: 1568
  REFERENCES: 446
  DEFINES: 1
  HAS_TOPIC: 1243
  HAS_RULE: 1920
  BELONGS_TO: 1

Most Referenced Articles:
  penal_code - Article 77: 24 references
  penal_code - Article 17: 18 references
  anti_drugs - Article 37: 18 references
  penal_code - Article 87: 17 references
  penal_code - Article 82: 15 references

Most Common Topics:
  Penalty: 464 articles
  Investigation: 155 articles
  Appeal: 80 articles
  Trial: 63 article